# HU-EEQI v6 · Notebook 05 — Robustez das tendências (resposta aos revisores do Mann–Kendall)

Rode **depois** dos notebooks 02 (índice) e 04 (tendências). Todos os números de tabelas, figuras e texto saem das saídas finais, sem edição manual.

| Crítica | O que o notebook faz | Saídas (em `Meu Drive/hu_eeqi_v6/robustez_05/`) |
|---|---|---|
| 1 Autocorrelação | Lag-1 dos resíduos de Sen. Testes (a) MK original, (b) MK com empates, (c) Hamed–Rao 1998, (d) TFPW Yue–Wang 2002. % significativo, perda em p.p., concordância e kappa. Validação contra `pymannkendall`. | `autocorrelation_by_unit.csv`, `tests_by_unit.csv`, `agreement_kappa.csv`, `validation_pymannkendall.csv`, `fig_r1_hist.png` |
| 2 Testes múltiplos | Benjamini–Hochberg por capital sobre o p do teste principal. | `class_hr_fdr` nos resultados; `fdr_thresholds_by_city.csv` (notebook 04) |
| 3 Sazonalidade e homogeneidade | Variante com estação fixa (a estação padrão de 3 meses mais seca em cada capital, pela climatologia ERA5-Land 2000–2025). Variante sem harmonização. Pettitt em séries da unidade e de áreas estáveis (MapBiomas). n de observações por ano e unidade. | `season_by_city.csv`, `nobs_by_unit_year.csv`, `pettitt.csv` |
| 4–5 Método, exatidão e limitações | Tabela de sensibilidade consolidada, conclusões robustas, texto em inglês (Métodos, Resultados, Limitações, Introdução) e respostas ao revisor. | `sensitivity_table.csv`, `robust_conclusions.csv`, `manuscript_text_EN.md`, `reviewer_responses_EN.md`, `classes_points_*.geojson`, `fig_classes_*.png` |

**Desenho da amostra.** Hamed–Rao e TFPW também são calculados por pixel no notebook 04 (mapas). Mesmo assim, toda a comparação entre testes e variantes usa o **mesmo painel**: amostra aleatória estratificada por capital, com `N_PANEL_PER_CITY` pixels e semente `PANEL_SEED + índice da capital`. A base usa a série **preenchida** do notebook 02b (sem lacunas). As variantes recebem, nos pontos, a mesma cascata de preenchimento em numpy (`fill_series_np`). Como sensibilidade, tudo é repetido no subconjunto `base_nofill`: pixels sem nenhum ano preenchido. As variantes (sem harmonização, estação fixa) são recalculadas só nos pontos do painel, com a **mesma calibração** da versão de base. Assim, muda apenas o fator testado. A avaliação dos testes corrigidos é **a posteriori**, e o texto declara isso.

In [ ]:
# @title Configuração HU-EEQI v6  (mantenha IDÊNTICA em todos os notebooks)
import os
# ---- Projeto, conta e assets --------------------------------------------------
# Edite os valores aqui. Se existir uma variável de ambiente (ou um Secret do Colab,
# ícone de chave) com o nome indicado no comentário, ela VENCE o valor digitado.
EE_PROJECT        = ""                                  # @param {type:"string"}
EE_ACCOUNT        = ""                  # @param {type:"string"}
CITIES_ASSET      = "projects/rnvuefsppgm/assets/Pol_capitais"        # @param {type:"string"}
CITY_NAME_PROP    = "NM_MUN"                                          # @param {type:"string"}
# Destinos: projeto Cloud mapbiomas-caatinga-cloud04 (formato projects/<projeto>/assets/<nome>)
RAW_COLLECTION    = "projects/mapbiomas-caatinga-cloud04/assets/HU-EEQI_v6_raw"      # @param {type:"string"}
OUT_COLLECTION    = "projects/mapbiomas-caatinga-cloud04/assets/HU-EEQI_v6"          # @param {type:"string"}
FILLED_COLLECTION = "projects/mapbiomas-caatinga-cloud04/assets/HU-EEQI_v6_filled"   # @param {type:"string"}
TRENDS_COLLECTION = "projects/mapbiomas-caatinga-cloud04/assets/TRENDS_HU-EEQI_v6"   # @param {type:"string"}
GDP_ASSET         = "projects/sat-io/open-datasets/GRIDDED_HDI_GDP/adm1_gdp_perCapita_1990_2022"   # @param {type:"string"}
MAPBIOMAS_ASSET   = "projects/mapbiomas-public/assets/brazil/lulc/collection9/mapbiomas_collection90_integration_v1"   # @param {type:"string"}
CENSUS_2010_ASSET = ""   # @param {type:"string"}
CENSUS_2022_ASSET = ""   # @param {type:"string"}
# variável de ambiente / Secret -> variável acima
_ENV_NAMES = {"EE_PROJECT": "EE_PROJECT", "HU_EE_ACCOUNT": "EE_ACCOUNT",
              "HU_CITIES_ASSET": "CITIES_ASSET", "HU_EEQI_V6_COLLECTION": "OUT_COLLECTION",
              "HU_EEQI_V6_RAW_COLLECTION": "RAW_COLLECTION",
              "HU_EEQI_V6_FILLED_COLLECTION": "FILLED_COLLECTION",
              "HU_EEQI_V6_TRENDS_COLLECTION": "TRENDS_COLLECTION", "HU_GDP_ASSET": "GDP_ASSET",
              "HU_MAPBIOMAS_ASSET": "MAPBIOMAS_ASSET",
              "HU_CENSUS_2010_ASSET": "CENSUS_2010_ASSET", "HU_CENSUS_2022_ASSET": "CENSUS_2022_ASSET"}

def _from_env_or_secret(env_name):
    v = os.environ.get(env_name)
    if v:
        return v
    try:
        from google.colab import userdata
        return userdata.get(env_name) or ""
    except Exception:          # fora do Colab, sem Secret ou sem acesso ao Secret
        return ""

for _env, _var in _ENV_NAMES.items():
    _v = _from_env_or_secret(_env)
    if _v:
        globals()[_var] = _v.strip()
for _var in ["EE_PROJECT", "CITIES_ASSET", "OUT_COLLECTION"]:
    if not globals()[_var]:
        raise ValueError(f"Preencha {_var} na célula de configuração.")
print("Projeto :", EE_PROJECT, "| conta:", EE_ACCOUNT)
print("Capitais:", CITIES_ASSET, "| pilha bruta:", RAW_COLLECTION, "| índice:", OUT_COLLECTION)
print("Série preenchida:", FILLED_COLLECTION, "| tendências:", TRENDS_COLLECTION)
# assets que a célula de autenticação testa (leitura) antes de qualquer task
READ_ASSETS = {"capitais": CITIES_ASSET, "PIB (V_SOURCE='gdp_adm1')": GDP_ASSET,
               "MapBiomas (notebooks 03/05)": MAPBIOMAS_ASSET}

# ---- saídas no Drive -----------------------------------------------------------
DRIVE_FOLDER    = "hu_eeqi_v6"            # pasta na raiz do Meu Drive (tabelas e JSON)
CALIB_JSON_NAME = "calibration_v6.json"   # DRIVE_DIR e CALIB_JSON são definidos na autenticação
OLD_CALIB_REL   = "hu_eeqi/calibration2.json"   # calibração v3 (só para comparar), relativo ao Meu Drive

# ---- série temporal ----------------------------------------------------------
YEAR_START, YEAR_END = 2000, 2025
YEARS = list(range(YEAR_START, YEAR_END + 1))

# ---- escalas -----------------------------------------------------------------
# Cada camada fica na resolução ORIGINAL (Landsat 30 m, ERA5-Land ~9 km, MAIAC 1 km,
# GHS-POP 100 m, PIB adm1): nenhuma reprojeção, agregação ou interpolação. O índice é
# calculado pixel a pixel na grade de exportação (30 m, a do Landsat).
EQ_SCALE       = 30       # escala de exportação = resolução do Landsat (m)
EXPORT_CRS     = "EPSG:4326"   # mesmo CRS de exportação da v3
AQ_STATS_SCALE = 1000     # só p/ a estatística escalar da cidade (mediana de AQ)
V_STATS_SCALE  = 1000     # só p/ os percentis p2/p98 de V na cidade

# ---- domínio de análise (FIXO no tempo) ---------------------------------------
APPLY_URBAN_MASK = True   # True = união GHS-SMOD >= URBAN_SMOD_MIN em 2000-2025
URBAN_SMOD_MIN   = 21     # 21 suburbano, 22 semi-denso, 23 denso, 30 centro urbano
WATER_OCC_MAX    = 50     # % de ocorrência JRC-GSW acima da qual o pixel é água

# ---- [1] harmonização TM/ETM+ -> OLI (Roy et al., 2016) -----------------------
HARMONIZE  = True
ROY_METHOD = "OLS"        # "OLS" ou "RMA" (Tabela 2 de Roy et al., 2016)
L7_POLICY  = "no_oli_years"   # "no_oli_years": ETM+ só até 2012 | "always"

# ---- [2] qualidade Landsat -----------------------------------------------------
ST_QA_MAX_K      = None   # incerteza máxima da LST (K); None = sem limite. v6: com 2 K nenhum pixel
                         # passava (ST_QA fica em 2,5–7 K nas capitais); a LST é só diagnóstico
MIN_OBS          = 3      # mínimo de observações limpas no ano por pixel
COMPOSITE_MONTHS = None   # ex.: (6, 9) = jun-set (estação seca); None = ano inteiro
CITY_MONTHS      = {}     # janela por cidade, ex.: {"Manaus": (7, 10)}

# ---- [4]/[8] HEAT: temperatura do ar e ponto de orvalho (ERA5-Land horário) ---
HEAT_METHOD      = "nws_hi"           # "nws_hi" (Rothfusz/NWS) | "steadman_at"
HEAT_LOCAL_HOURS = (12, 16)           # janela da tarde, em hora LOCAL
HEAT_STAT        = "median_daily_max" # mediana anual do máximo diário | "p95_daily_max"
HEAT_XC_THRESH_C = 32.2               # limiar NWS "extreme caution" (90 °F)
# HEAT calculado 1 vez por célula do ERA5-Land (0,1°) e não 1 vez por pixel de 30 m.
# Mesmo valor (testado: diferença 0): cada pixel de 30 m já recebia o valor da célula
# em que cai. Sem reamostragem nem agregação; só evita repetir a mesma conta ~90 mil vezes.
NATIVE_GRID_EVAL = True
# v6: célula do ERA5-Land sem dado (o ERA5-Land mascara o mar; ~7% de Aracaju) recebe a média das
# células vizinhas imediatas com dado (3×3, grade nativa, mesmo ano); marcada na banda HEAT_filled.
HEAT_FILL_NEIGHBORS = True

# ---- AQ ------------------------------------------------------------------------
AQ_SOURCE          = "maiac"  # "maiac" (AOD 550 nm) | "pm25" (coleção abaixo)
AQ_REQUIRE_BEST_QA = True     # MAIAC: céu claro + sem adjacência + QA de AOD "best"
AOD_MIN_OBS        = 10       # mínimo de retrievals válidos no ano por pixel
AQ_FILL            = "city_median"  # lacuna -> mediana do ano na cidade | "none" (fica sem dado)
PM25_COLLECTION    = None     # ex.: ImageCollection anual de PM2.5 (se tiver acesso)
PM25_BAND          = None

# ---- [3] população -------------------------------------------------------------
POP_SOURCE = "ghs"                        # "ghs" (GHS-POP R2023A) | "worldpop"
GHS_POP_IC = "JRC/GHSL/P2023A/GHS_POP"
POP_W_EPS  = 1.0     # hab por célula de 100 m somados ao peso: área vazia -> peso ~uniforme

# ---- vulnerabilidade / justiça -------------------------------------------------
USE_JUSTICE     = True
V_SOURCE        = "gdp_adm1"   # "census" (RECOMENDADO) | "gdp_adm1" (GDP_ASSET acima)
GDP_YEAR_START, GDP_YEAR_END = 1990, 2022
CENSUS_INCOME_PROP = ""   # nome da propriedade de renda nos assets de setores censitários
V_CENSUS = {y: {"asset": a, "prop": CENSUS_INCOME_PROP}
            for y, a in ((2010, CENSUS_2010_ASSET), (2022, CENSUS_2022_ASSET)) if a}
V_MIN_REL_RANGE = 1e-3          # (p98-p2)/|p98| mínimo p/ J ser informativo
J_FALLBACK      = "renormalize" # J degenerado: "renormalize" | "to_eq"

# ---- fusão  HU-EEQI = EQ^a · E^b · J^c ----------------------------------------
ALPHA, BETA, GAMMA = 0.6, 0.2, 0.2
WIN_RADIUS_M = 2000
NI_FLOOR     = 0.01

# ---- [7] calibração --------------------------------------------------------------
BOUNDS_PCT    = (1, 99)     # limites agrupados (todas as cidades e todos os anos)
WEIGHT_METHOD = "critic"    # "critic" | "ewm" | "pca" | "equal"

DESCRIPTORS = ["NDVI", "Wet", "NDBSI", "HEAT", "AQ"]
DIRECTION   = {"NDVI": "benefit", "Wet": "benefit", "NDBSI": "cost",
               "HEAT": "cost", "AQ": "cost"}

# ---- [6] exportação --------------------------------------------------------------
EXPORT_DIAGNOSTICS = True   # descritores brutos + QA (obrigatório p/ os notebooks 03 e 05)
SKIP_EXISTING      = True   # pula assets já exportados (permite retomar)
VERSION = "v6"

# ---- painel fixo de pixels (notebooks 03 e 05) -----------------------------------
N_PANEL_PER_CITY = 400      # pixels por capital (amostra estratificada por unidade)
PANEL_SEED       = 7        # semente base; capital i usa PANEL_SEED + i
MB_YEARS         = (2000, 2023)   # anos MapBiomas usados para definir áreas estáveis

# ---- tendências (notebooks 04 e 05) ------------------------------------------------
ALPHA_SIG     = 0.05        # nível de significância (o mesmo p < 0,05 do Trends v3)
FDR_Q         = ALPHA_SIG   # nível q de Benjamini–Hochberg (item 2)
FDR_HIST_BINS = 50000       # resolução do histograma de p em [0, ALPHA_SIG] (BH conservador)
R1_THRESHOLD  = 0.3         # limiar de autocorrelação lag-1 pedido na revisão

# ---- [9] preenchimento de lacunas (notebook 02b; usado nos 04 e 05) -----------------
# Lacuna = pixel do domínio sem valor no ano (nuvem/sombra, < MIN_OBS observações limpas, ano inteiro
# sem imagem, HEAT/AQ/LST sem dado). Cascata pixel a pixel e banda a banda, sobre a série anual JÁ
# exportada, em TODAS as bandas contínuas (índice, componentes, descritores, LST, POP e V):
#   fill_type 0 = observado
#   fill_type 1 = interpolação linear no tempo entre o último ano com dado antes e o primeiro
#                 depois (lacuna de 1 ano = média dos dois; vale também para o ano inteiro vazio)
#   fill_type 2 = ponta da série (só um lado tem dado): ano com dado mais próximo
#   fill_type 3 = pixel sem dado em nenhum ano: média dos pixels com valor no mesmo ano numa janela
#                 de WIN_RADIUS_M (a mesma janela de E e J)
#   fill_type 4 = nenhum pixel com valor nessa janela: mediana da capital no ano
# filled_mask: bit i = banda FILL_BANDS[i] preenchida (não observada) naquele pixel-ano.
FILL_BANDS      = ["HU_EEQI", "EQ", "E_local", "J_local",
                   "NI_NDVI", "NI_Wet", "NI_NDBSI", "NI_HEAT", "NI_AQ",
                   "NDVI", "Wet", "NDBSI", "HEAT", "AQ", "LST", "LST_anom", "HEAT_ndays_xc", "POP", "V"]
FILL_KEY_BAND   = "HU_EEQI"   # fill_type e gap_years descrevem esta banda
# contagens e marcas copiadas do ano (0 = sem observação/sem marca naquele ano); urban é recalculada
FILL_COPY_BANDS = ["n_obs", "sensor_flag", "n_TM", "n_ETM", "n_OLI", "n_obs_thermal", "n_aod",
                   "AQ_filled", "HEAT_filled", "NI_clamped"]
FILL_REPORT_GAP = 3           # resumo: pixels preenchidos com lacuna maior que 3 anos
TREND_SOURCE    = "filled"    # 04/05: "filled" = série preenchida (02b) | "observed" = só anos observados


In [ ]:
# @title Autenticar (conta EE_ACCOUNT), iniciar o Earth Engine no projeto e montar o Drive
import json, os, subprocess, sys

def _pip(pkg, module=None):
    """Instala um pacote só se ainda não estiver disponível (Colab ou máquina local)."""
    try:
        __import__(module or pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

_pip("earthengine-api", "ee")
import ee
try:
    from google.colab import auth as _colab_auth, drive as _colab_drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

def _active_account():
    """Melhor esforço: e-mail da credencial ativa (gcloud; senão, userinfo). '' se não souber."""
    try:
        r = subprocess.run(["gcloud", "auth", "list", "--filter=status:ACTIVE",
                            "--format=value(account)"], capture_output=True, text=True, timeout=60)
        lines = [l.strip() for l in r.stdout.splitlines() if "@" in l]
        if lines:
            return lines[0]
    except Exception:
        pass
    try:
        import google.auth, google.auth.transport.requests, requests
        creds, _ = google.auth.default()
        creds.refresh(google.auth.transport.requests.Request())
        r = requests.get("https://www.googleapis.com/oauth2/v3/userinfo",
                         headers={"Authorization": "Bearer " + creds.token}, timeout=30)
        return r.json().get("email", "")
    except Exception:
        return ""

if IN_COLAB:
    # 1) Pop-up do Colab: escolha a conta EE_ACCOUNT. O Earth Engine usa essa mesma credencial
    #    (ee.Initialize lê as "application default credentials" criadas aqui).
    _colab_auth.authenticate_user()
    # Um arquivo de credenciais do Earth Engine deixado no runtime (ee.Authenticate() anterior,
    # talvez com outra conta) teria prioridade sobre a conta escolhida no pop-up: remove.
    _old_cred = os.path.expanduser("~/.config/earthengine/credentials")
    if os.path.exists(_old_cred):
        os.remove(_old_cred)
        print("Credencial antiga do Earth Engine removida do runtime.")
    ACCOUNT = _active_account()
    if ACCOUNT and EE_ACCOUNT and ACCOUNT.lower() != EE_ACCOUNT.lower():
        raise RuntimeError(f"Conta autenticada = {ACCOUNT}, esperada = {EE_ACCOUNT}. Vá em "
                           "Runtime > Disconnect and delete runtime e autentique com a conta certa "
                           "(ou mude EE_ACCOUNT na configuração).")
    print("Conta autenticada:", ACCOUNT or "(não foi possível confirmar o e-mail)")
else:
    # Máquina local: credenciais do `earthengine authenticate` (ou pop-up na 1ª vez)
    ACCOUNT = ""
    try:
        ee.Initialize(project=EE_PROJECT)
    except Exception:
        ee.Authenticate()
# 2) Projeto: falha AQUI (e não no meio das exportações) se a conta não puder usá-lo
try:
    ee.Initialize(project=EE_PROJECT)
    ee.Number(1).getInfo()
except Exception as e:
    raise PermissionError(
        f"A conta {ACCOUNT or 'autenticada'} não conseguiu usar o projeto {EE_PROJECT}. "
        "Confira o nome do projeto e peça, nesse projeto, os papéis "
        "'Service Usage Consumer' e 'Earth Engine Resource Writer'.\nDetalhe: " + str(e)[:300])
print(f"Earth Engine OK | projeto {EE_PROJECT} | tasks criadas por {ACCOUNT or 'esta conta'}")
# 3) Leitura dos assets de entrada
for _nome, _aid in READ_ASSETS.items():
    if not _aid:
        continue
    try:
        ee.data.getAsset(_aid)
        print("  leitura OK :", _nome, "->", _aid)
    except Exception as e:
        raise PermissionError(f"Sem acesso de leitura ao asset '{_nome}': {_aid}\n"
                              f"Detalhe: {str(e)[:200]}")
# 4) Drive (Colab) ou pasta local. Monte com a MESMA conta: os exports toDrive vão para ela.
if IN_COLAB:
    _colab_drive.mount("/content/drive")
    DRIVE_ROOT = "/content/drive/MyDrive"
else:
    DRIVE_ROOT = os.path.abspath(os.environ.get("HU_DRIVE_ROOT", "."))
DRIVE_DIR      = f"{DRIVE_ROOT}/{DRIVE_FOLDER}"
CALIB_JSON     = f"{DRIVE_DIR}/{globals().get('CALIB_JSON_NAME', 'calibration.json')}"
OLD_CALIB_JSON = f"{DRIVE_ROOT}/{globals().get('OLD_CALIB_REL', 'hu_eeqi/calibration2.json')}"
os.makedirs(DRIVE_DIR, exist_ok=True)
print("Pasta de trabalho:", DRIVE_DIR)

In [ ]:
# @title Biblioteca HU-EEQI v6  (não editar — idêntica em todos os notebooks)
import ee, json, math, re, unicodedata

# ============================================================================
# 0) Constantes de dados
# ============================================================================
OPT_BANDS   = ["BLUE", "GREEN", "RED", "NIR", "SWIR1", "SWIR2"]
SENSOR_CODE = {"TM": 1, "ETM": 2, "OLI": 4, "OLI2": 8}      # bits do sensor_flag
LS_COLLECTIONS = [("LANDSAT/LT05/C02/T1_L2", "TM"),
                  ("LANDSAT/LE07/C02/T1_L2", "ETM"),
                  ("LANDSAT/LC08/C02/T1_L2", "OLI"),
                  ("LANDSAT/LC09/C02/T1_L2", "OLI2")]
SENSOR_YEARS = {"TM": (1984, 2012), "ETM": (1999, 2024),
                "OLI": (2013, 2100), "OLI2": (2021, 2100)}
SR_SRC = {"TM":   ["SR_B1", "SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B7"],
          "ETM":  ["SR_B1", "SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B7"],
          "OLI":  ["SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B6", "SR_B7"],
          "OLI2": ["SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B6", "SR_B7"]}
ST_SRC = {"TM": "ST_B6", "ETM": "ST_B6", "OLI": "ST_B10", "OLI2": "ST_B10"}

# [1] Roy et al. (2016) RSE 185:57-70, Tabela 2 — transformação ETM+ -> OLI,
#     refletância de superfície, ordem BLUE, GREEN, RED, NIR, SWIR1, SWIR2.
#     Aplicada também ao TM (assumido equivalente ao ETM+). Confira na tabela
#     original antes da submissão.
ROY_COEF = {
    "OLS": {"slope": [0.8474, 0.8483, 0.9047, 0.8462, 0.8937, 0.9071],
            "itcp":  [0.0003, 0.0088, 0.0061, 0.0412, 0.0254, 0.0172]},
    "RMA": {"slope": [0.9785, 0.9542, 0.9825, 1.0073, 1.0171, 0.9949],
            "itcp":  [-0.0095, -0.0016, -0.0022, -0.0021, -0.0030, 0.0029]},
}

# Tasseled-cap Wetness (Crist, 1985; refletância). Após a harmonização, todos os
# sensores estão na escala OLI, então UM único conjunto é aplicado à série toda.
TCW = {"BLUE": 0.1509, "GREEN": 0.1973, "RED": 0.3279, "NIR": 0.3406,
       "SWIR1": -0.7112, "SWIR2": -0.4572}

SMOD_IC     = "JRC/GHSL/P2023A/GHS_SMOD_V2-0"
GSW_IMG     = "JRC/GSW1_4/GlobalSurfaceWater"
ERA5_HOURLY = "ECMWF/ERA5_LAND/HOURLY"
MAIAC_IC    = "MODIS/061/MCD19A2_GRANULES"
WORLDPOP_IC = "WorldPop/GP/100m/pop"

CORE_BANDS = ["HU_EEQI", "EQ", "E_local", "J_local",
              "NI_NDVI", "NI_Wet", "NI_NDBSI", "NI_HEAT", "NI_AQ",
              "n_obs", "sensor_flag", "POP", "V", "urban", "domain"]
DIAG_BANDS = ["NDVI", "Wet", "NDBSI", "HEAT", "AQ", "NI_clamped",
              "n_TM", "n_ETM", "n_OLI", "n_obs_thermal", "LST", "LST_anom",
              "n_aod", "AQ_filled", "HEAT_ndays_xc", "HEAT_filled"]


# ============================================================================
# 1) Utilidades, cidades e domínio
# ============================================================================
def ascii_safe(s):
    a = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode()
    return re.sub(r"[^A-Za-z0-9.-]", "_", a)[:40] or "city"

def _setter(props):
    def f(feat):
        return feat.set(props)
    return f

def cities():
    return ee.FeatureCollection(CITIES_ASSET)

def city_contexts():
    """Lê nomes e longitudes de uma vez (2 getInfo) e monta o contexto por cidade."""
    fc = cities()
    names = fc.aggregate_array(CITY_NAME_PROP).getInfo()
    lons = (fc.map(lambda f: ee.Feature(None, {
                "lon": f.geometry().centroid(1000).coordinates().get(0)}))
              .aggregate_array("lon").getInfo())
    flist = fc.toList(fc.size())
    ctxs = []
    for i, (name, lon) in enumerate(zip(names, lons)):
        region = ee.Feature(flist.get(i)).geometry()
        ctxs.append({
            "idx": i, "name": name, "safe": ascii_safe(name), "region": region,
            # região com margem: evita efeito de borda nos focais E/J
            "region_buf": region.buffer(WIN_RADIUS_M + 1000, 100),   # margem p/ a janela de E/J
            "utc_offset": int(round(lon / 15.0)),    # hora SOLAR (lon/15): Brasília -3, Rio Branco -5, Recife -2
            "months": CITY_MONTHS.get(name, COMPOSITE_MONTHS),
        })
    return ctxs

def domain_mask():
    """Domínio FIXO: terra (JRC-GSW) ∩ união urbana GHS-SMOD 2000-2025."""
    land = ee.Image(GSW_IMG).select("occurrence").unmask(0).lt(WATER_OCC_MAX)
    if not APPLY_URBAN_MASK:
        return land.rename("domain")
    urb = (ee.ImageCollection(SMOD_IC)
           .filterDate(f"{YEAR_START}-01-01", f"{YEAR_END + 1}-01-01")
           .map(lambda i: i.select(0).gte(URBAN_SMOD_MIN))
           .max())
    return land.And(urb).rename("domain")

def urban_mask(year):
    """Urbano do ano (época SMOD mais próxima) — só informativo; o domínio é fixo."""
    e = min(max(int(5 * round(year / 5.0)), 1975), 2030)
    smod = ee.ImageCollection(SMOD_IC).filter(ee.Filter.calendarRange(e, e, "year")).first()
    return ee.Image(smod).select(0).gte(URBAN_SMOD_MIN).rename("urban")

def config_snapshot():
    """Parâmetros que mudam os descritores brutos: a calibração tem de bater com eles."""
    snap = {"HARMONIZE": HARMONIZE, "ROY_METHOD": ROY_METHOD, "L7_POLICY": L7_POLICY,
            "ST_QA_MAX_K": ST_QA_MAX_K, "MIN_OBS": MIN_OBS,
            "COMPOSITE_MONTHS": COMPOSITE_MONTHS, "CITY_MONTHS": CITY_MONTHS,
            "HEAT_METHOD": HEAT_METHOD, "HEAT_LOCAL_HOURS": HEAT_LOCAL_HOURS,
            "HEAT_STAT": HEAT_STAT, "HEAT_FILL_NEIGHBORS": HEAT_FILL_NEIGHBORS, "AQ_SOURCE": AQ_SOURCE,
            "AQ_REQUIRE_BEST_QA": AQ_REQUIRE_BEST_QA, "AOD_MIN_OBS": AOD_MIN_OBS,
            "AQ_FILL": AQ_FILL, "PM25_COLLECTION": PM25_COLLECTION,
            "APPLY_URBAN_MASK": APPLY_URBAN_MASK, "URBAN_SMOD_MIN": URBAN_SMOD_MIN,
            "WATER_OCC_MAX": WATER_OCC_MAX, "EQ_SCALE": EQ_SCALE,
            "ANNUAL_STAT": "median", "BOUNDS_PCT": BOUNDS_PCT,
            "DESCRIPTORS": DESCRIPTORS, "DIRECTION": DIRECTION}
    return json.loads(json.dumps(snap))

def check_calibration(calib):
    old = calib.get("config", {})
    cur = config_snapshot()
    diff = {k: {"calib": old.get(k), "atual": v} for k, v in cur.items() if old.get(k) != v}
    if diff:
        raise ValueError("Calibração incompatível com a configuração atual — rode o "
                         "notebook 01 de novo.\n" + json.dumps(diff, ensure_ascii=False, indent=1))
    print("Calibração compatível (criada em %s, n=%s amostras)."
          % (calib.get("created"), calib.get("n_samples")))


# ============================================================================
# 2) Landsat C2 L2 — [2] máscara QA, [1] harmonização, composição com n_obs
# ============================================================================
def _prep_landsat(img, sensor):
    qa = img.select("QA_PIXEL")
    # bits 0-4: fill, dilated cloud, cirrus, cloud, cloud shadow -> todos têm de ser 0
    clear = qa.bitwiseAnd(0b11111).eq(0)
    unsat = img.select("QA_RADSAT").eq(0)              # nenhuma banda saturada
    good  = clear.And(unsat)

    sr = img.select(SR_SRC[sensor], OPT_BANDS).multiply(0.0000275).add(-0.2)
    if HARMONIZE and sensor in ("TM", "ETM"):
        c = ROY_COEF[ROY_METHOD]
        sr = (sr.multiply(ee.Image.constant(c["slope"]))
                .add(ee.Image.constant(c["itcp"]))
                .rename(OPT_BANDS))
    # remove refletâncias fisicamente impossíveis (bordas de nuvem, sombra profunda)
    phys = sr.reduce(ee.Reducer.min()).gt(0).And(sr.reduce(ee.Reducer.max()).lt(1))
    sr = sr.updateMask(good.And(phys))

    lst = (img.select(ST_SRC[sensor]).multiply(0.00341802).add(149.0)
              .subtract(273.15).rename("LST"))
    st_unc = img.select("ST_QA").multiply(0.01)       # incerteza em K
    lst = lst.updateMask(good).updateMask(st_unc.gt(0))
    if ST_QA_MAX_K is not None:
        lst = lst.updateMask(st_unc.lte(ST_QA_MAX_K))

    sensor_band = (ee.Image.constant(SENSOR_CODE[sensor]).toInt()
                   .rename("sensor").updateMask(sr.select("NIR").mask()))
    out = sr.addBands(lst).addBands(sensor_band).set("sensor", sensor)
    return ee.Image(out.copyProperties(img, ["system:time_start"]))

def _prep_fn(sensor):
    def f(im):
        return _prep_landsat(im, sensor)
    return f

def _date_filter(col, year, months):
    col = col.filterDate(f"{year}-01-01", f"{year + 1}-01-01")
    if months:
        # calendarRange com início > fim faz a volta do ano (ex.: (11, 2))
        col = col.filter(ee.Filter.calendarRange(int(months[0]), int(months[1]), "month"))
    return col

def landsat_collection(year, region, months):
    cols = []
    for cid, sensor in LS_COLLECTIONS:
        y0, y1 = SENSOR_YEARS[sensor]
        if not (y0 <= year <= y1):
            continue
        if sensor == "ETM" and L7_POLICY == "no_oli_years" and year >= 2013:
            continue     # SLC-off + deriva de órbita: só usa ETM+ quando não há OLI
        c = _date_filter(ee.ImageCollection(cid).filterBounds(region), year, months)
        cols.append(c.map(_prep_fn(sensor)))
    col = cols[0]
    for c in cols[1:]:
        col = col.merge(c)
    return col

def _count_eq(code):
    def f(i):
        return i.eq(code)
    return f

def landsat_composite(year, ctx):
    """Mediana harmonizada + bandas de QA (n_obs, sensores, LST diagnóstica)."""
    col = landsat_collection(year, ctx["region_buf"], ctx["months"])
    n_obs = col.select("NIR").count().rename("n_obs")
    opt = col.select(OPT_BANDS).median().updateMask(n_obs.gte(MIN_OBS))
    sens = col.select("sensor")
    sensor_flag = sens.reduce(ee.Reducer.bitwiseOr()).rename("sensor_flag")
    n_tm  = sens.map(_count_eq(1)).sum().unmask(0).rename("n_TM")
    n_etm = sens.map(_count_eq(2)).sum().unmask(0).rename("n_ETM")
    n_oli = sens.map(lambda i: i.gte(4)).sum().unmask(0).rename("n_OLI")
    lst   = col.select("LST").median().rename("LST")
    n_th  = col.select("LST").count().rename("n_obs_thermal")
    qa = ee.Image.cat([n_obs.unmask(0), sensor_flag, n_tm, n_etm, n_oli, n_th, lst])
    return opt.clip(ctx["region_buf"]), qa.clip(ctx["region_buf"])

def descriptors_optical(comp):
    nir, red = comp.select("NIR"), comp.select("RED")
    swir1, blue, green = comp.select("SWIR1"), comp.select("BLUE"), comp.select("GREEN")
    ndvi = nir.subtract(red).divide(nir.add(red)).rename("NDVI")
    wet = comp.select("BLUE").multiply(TCW["BLUE"])
    for b in ["GREEN", "RED", "NIR", "SWIR1", "SWIR2"]:
        wet = wet.add(comp.select(b).multiply(TCW[b]))
    wet = wet.rename("Wet")
    # NDBSI = (IBI + SI) / 2  (Xu, 2008; Xu, 2013)
    t1 = swir1.multiply(2).divide(swir1.add(nir))
    t2 = nir.divide(nir.add(red)).add(green.divide(green.add(swir1)))
    ibi = t1.subtract(t2).divide(t1.add(t2))
    si = swir1.add(red).subtract(nir.add(blue)).divide(swir1.add(red).add(nir.add(blue)))
    ndbsi = ibi.add(si).divide(2).rename("NDBSI")
    return ee.Image.cat([ndvi, wet, ndbsi])


# ============================================================================
# 3) [4]/[8] HEAT — índice de calor com TEMPERATURA DO AR (ERA5-Land horário)
#     Nada de LST aqui. T e Td vêm do mesmo instante; RH via Magnus.
# ============================================================================
def _es_hpa(t):
    """Pressão de saturação (hPa), Magnus — Alduchov & Eskridge (1996); t em °C."""
    return t.expression("6.1094 * exp(17.625 * T / (T + 243.04))", {"T": t})

def heat_index_nws(t_c, rh):
    """Heat index NWS (Rothfusz 1990 + ajustes NWS). Entrada °C e %, saída °C."""
    T = t_c.multiply(1.8).add(32)                     # °F
    simple = T.expression("0.5 * (T + 61.0 + (T - 68.0) * 1.2 + RH * 0.094)",
                          {"T": T, "RH": rh})
    roth = T.expression(
        "-42.379 + 2.04901523*T + 10.14333127*RH - 0.22475541*T*RH"
        " - 0.00683783*T*T - 0.05481717*RH*RH + 0.00122874*T*T*RH"
        " + 0.00085282*T*RH*RH - 0.00000199*T*T*RH*RH", {"T": T, "RH": rh})
    adj_dry = T.expression("((13 - RH) / 4) * sqrt((17 - abs(T - 95)) / 17)",
                           {"T": T, "RH": rh})
    adj_wet = T.expression("((RH - 85) / 10) * ((87 - T) / 5)", {"T": T, "RH": rh})
    roth = roth.where(rh.lt(13).And(T.gte(80)).And(T.lte(112)), roth.subtract(adj_dry))
    roth = roth.where(rh.gt(85).And(T.gte(80)).And(T.lte(87)), roth.add(adj_wet))
    hi_f = simple.where(simple.add(T).divide(2).gte(80), roth)
    return hi_f.subtract(32).divide(1.8)

def _hourly_heat(img):
    t  = img.select("temperature_2m").subtract(273.15)
    td = img.select("dewpoint_temperature_2m").subtract(273.15)
    e  = _es_hpa(td)                                  # pressão de vapor real (hPa)
    rh = e.divide(_es_hpa(t)).multiply(100).clamp(0, 100)
    if HEAT_METHOD == "steadman_at":
        # Steadman (1994), temperatura aparente à sombra: AT = Ta + 0.33e - 0.70ws - 4.00
        ws = img.select("u_component_of_wind_10m").hypot(img.select("v_component_of_wind_10m"))
        hi = t.add(e.multiply(0.33)).subtract(ws.multiply(0.70)).subtract(4.00)
    else:
        hi = heat_index_nws(t, rh)
    return ee.Image(hi.rename("HI").copyProperties(img, ["system:time_start"]))

def _day_key(img):
    return img.set("day", ee.Date(img.get("system:time_start")).format("YYYY-MM-dd"))

def _clip_to(geom):
    def f(img):
        return img.clip(geom)
    return f

def _daily_max(feat):
    hours = ee.ImageCollection.fromImages(feat.get("hours"))
    return hours.max().set("system:time_start", feat.get("system:time_start"))

def exact_median(ic):
    """Mediana EXATA por pixel (ordena o vetor de valores válidos). O ee.Reducer.median() é
    aproximado (histograma): no HEAT de Aracaju errou 0,02 °C."""
    arr = ic.toArray().arrayProject([0]).arraySort()
    n = arr.arrayLength(0)
    ok = n.gt(0)
    arr = arr.updateMask(ok)
    lo = arr.arrayGet(n.subtract(1).divide(2).floor().max(0).toInt())
    hi = arr.arrayGet(n.divide(2).floor().min(n.subtract(1)).max(0).toInt())
    return lo.add(hi).divide(2).updateMask(ok)

def heat_layer(year, ctx):
    """MEDIANA anual (ou p95) do máximo diário do índice de calor na janela da tarde.
    - Resolução ORIGINAL do ERA5-Land (~9 km): sem reprojeção nem interpolação; cada pixel
      de saída recebe o valor da célula ERA5 em que cai (vizinho mais próximo).
    - Máximo diário por JOIN (cada dia junta as suas horas). A versão anterior fazia 365
      filterDate dentro de um map e estourava a memória."""
    h0, h1 = HEAT_LOCAL_HOURS
    u0, u1 = h0 - ctx["utc_offset"], h1 - ctx["utc_offset"]   # local -> UTC
    if not (0 <= u0 <= u1 <= 23):
        raise ValueError(f"Janela UTC inválida {u0}-{u1} para {ctx['name']}")
    box = ctx["region_buf"].bounds(1000)
    if HEAT_FILL_NEIGHBORS:
        box = box.buffer(15000, 1000).bounds(1000)   # + 1 célula ERA5 (~11 km): vizinhas
    era = (ee.ImageCollection(ERA5_HOURLY)
           .filterDate(f"{year}-01-01", f"{year + 1}-01-01")
           .select(["temperature_2m", "dewpoint_temperature_2m",
                    "u_component_of_wind_10m", "v_component_of_wind_10m"]))
    hourly = (era.filter(ee.Filter.calendarRange(u0, u1, "hour"))
              .map(_clip_to(box)).map(_hourly_heat).map(_day_key))
    first = hourly.filter(ee.Filter.calendarRange(u0, u0, "hour"))   # 1 imagem por dia
    joined = ee.Join.saveAll("hours").apply(first, hourly,
                                            ee.Filter.equals(leftField="day", rightField="day"))
    daily = ee.ImageCollection(joined.map(_daily_max))
    if HEAT_STAT == "p95_daily_max":
        heat = daily.reduce(ee.Reducer.percentile([95])).rename("HEAT")
    else:                                     # "median_daily_max" (padrão)
        heat = exact_median(daily).rename("HEAT")   # ee.Reducer.median() é aproximado
    ndays_xc = daily.map(lambda i: i.gte(HEAT_XC_THRESH_C)).sum().rename("HEAT_ndays_xc")
    nat = ee.Image(era.first()).select("temperature_2m").projection()
    heat_flag = ee.Image.constant(0).rename("HEAT_filled")
    if HEAT_FILL_NEIGHBORS:
        # célula sem dado <- média das células vizinhas imediatas com dado (3×3), grade nativa, mesmo ano
        # (skipMasked=False — senão a saída fica mascarada justamente onde a célula central não tem dado)
        K1 = ee.Kernel.square(radius=1, units="pixels")
        h0, n0 = heat.reproject(nat), ndays_xc.reproject(nat)
        hN = h0.reduceNeighborhood(reducer=ee.Reducer.mean(), kernel=K1, skipMasked=False).reproject(nat)
        nN = n0.reduceNeighborhood(reducer=ee.Reducer.mean(), kernel=K1, skipMasked=False).reproject(nat)
        heat_flag = h0.mask().Not().And(hN.mask()).reproject(nat).rename("HEAT_filled")
        heat, ndays_xc = h0.unmask(hN).rename("HEAT"), n0.unmask(nN).rename("HEAT_ndays_xc")
    if NATIVE_GRID_EVAL or HEAT_FILL_NEIGHBORS:
        # conta feita 1 vez por célula ERA5-Land (0,1°), não por pixel de 30 m: mesmo valor,
        # pois cada pixel de 30 m já recebia o valor da célula em que cai (vizinho mais próximo)
        heat, ndays_xc = heat.reproject(nat), ndays_xc.reproject(nat)
    return heat, ndays_xc, heat_flag


# ============================================================================
# 4) AQ — MAIAC com QA, sem "zero = ar perfeito"
# ============================================================================
def _maiac_prep(img):
    aod = img.select("Optical_Depth_055").multiply(0.001)
    if AQ_REQUIRE_BEST_QA:
        qa = img.select("AOD_QA")
        clear = qa.bitwiseAnd(7).eq(1)                  # bits 0-2 = 001 (céu claro)
        adj   = qa.rightShift(5).bitwiseAnd(7).eq(0)    # bits 5-7 = 000 (sem adjacência)
        best  = qa.rightShift(8).bitwiseAnd(15).eq(0)   # bits 8-11 = 0000 (best quality)
        aod = aod.updateMask(clear.And(adj).And(best))
    return aod.rename("AOD")

def aq_layer(year, ctx):
    """AQ = MEDIANA anual na resolução ORIGINAL (MAIAC 1 km), sem reprojeção. Pixels com
    menos de AOD_MIN_OBS retrievals ficam sem dado; AQ_FILL decide o que fazer com eles."""
    s, e = f"{year}-01-01", f"{year + 1}-01-01"
    if AQ_SOURCE == "pm25":
        if not PM25_COLLECTION:
            raise ValueError("AQ_SOURCE='pm25' exige PM25_COLLECTION e PM25_BAND.")
        aq = ee.ImageCollection(PM25_COLLECTION).filterDate(s, e).select(PM25_BAND).median()
        n = ee.Image.constant(1).rename("n_aod")
    else:
        col = (ee.ImageCollection(MAIAC_IC).filterDate(s, e)
               .filterBounds(ctx["region_buf"]).map(_maiac_prep))
        n = col.count().rename("n_aod")
        aq = col.median().updateMask(n.gte(AOD_MIN_OBS))  # sem dado -> MASCARADO (não 0)
    aq = aq.rename("AQ")
    if AQ_FILL == "city_median":
        # lacuna -> mediana do ano na cidade (um número; nenhuma operação espacial)
        m = aq.reduceRegion(ee.Reducer.median(), ctx["region"], AQ_STATS_SCALE,
                            maxPixels=1e12, bestEffort=True).get("AQ")
        filled = ee.Image(ee.Algorithms.If(m, aq.unmask(ee.Image.constant(m)), aq)).rename("AQ")
    else:
        filled = aq
    flag = aq.mask().Not().And(filled.mask()).rename("AQ_filled")
    return filled, n.unmask(0), flag


# ============================================================================
# 5) Pilha bruta de descritores
# ============================================================================
def raw_stack(year, ctx):
    comp, ls_qa = landsat_composite(year, ctx)
    opt = descriptors_optical(comp)
    heat, heat_nd, heat_flag = heat_layer(year, ctx)
    aq, n_aod, aq_fill = aq_layer(year, ctx)
    raw = ee.Image.cat([opt, heat, aq]).select(DESCRIPTORS)
    qa = ee.Image.cat([ls_qa, heat_nd, heat_flag, n_aod, aq_fill])
    return raw, qa


# ============================================================================
# 6) Normalização (limites agrupados) + EQ (média geométrica ponderada)
# ============================================================================
def normalize_stack(raw, bounds):
    out, flags = [], ee.Image.constant(0)
    for i, name in enumerate(DESCRIPTORS):
        lo, hi = float(bounds[name]["min"]), float(bounds[name]["max"])
        x = raw.select(name)
        if DIRECTION[name] == "benefit":
            ni = x.subtract(lo).divide(hi - lo)
        else:
            ni = ee.Image.constant(hi).subtract(x).divide(hi - lo)
        clipped = ni.lt(NI_FLOOR).Or(ni.gt(1))
        flags = flags.add(clipped.multiply(2 ** i))          # bit i = descritor saturado
        out.append(ni.clamp(NI_FLOOR, 1.0).rename("NI_" + name))
    return ee.Image.cat(out), flags.rename("NI_clamped")

def compute_eq(ni, weights):
    tot = float(sum(weights[d] for d in DESCRIPTORS))
    logsum = ee.Image.constant(0.0)
    for d in DESCRIPTORS:
        logsum = logsum.add(ni.select("NI_" + d).log().multiply(float(weights[d]) / tot))
    return ee.Image(logsum).exp().clamp(NI_FLOOR, 1.0).rename("EQ")


# ============================================================================
# 7) [3] População — GHS-POP R2023A (100 m, resolução ORIGINAL) com interpolação log-linear
# ============================================================================
def _ghs_epoch(e):
    return ee.Image(ee.ImageCollection(GHS_POP_IC)
                    .filter(ee.Filter.calendarRange(e, e, "year")).first()).select(0)

def pop_provenance(year):
    if POP_SOURCE == "worldpop":
        return "observado" if 2000 <= year <= 2020 else "mantido_2020"
    return "observado" if year % 5 == 0 else "interpolado_loglinear"

def pop_year(year):
    """Habitantes por célula de 100 m (sem reprojeção nem agregação)."""
    if POP_SOURCE == "worldpop":
        y = min(max(year, 2000), 2020)
        return ee.Image(ee.ImageCollection(WORLDPOP_IC)
                        .filter(ee.Filter.eq("country", "BRA"))
                        .filter(ee.Filter.eq("year", y)).first()).select("population").rename("POP")
    lo = 5 * (year // 5)
    a = _ghs_epoch(lo)
    if year % 5 == 0:
        return a.rename("POP")
    hi = lo + 5
    b = _ghs_epoch(hi)
    f = (year - lo) / 5.0
    lin = a.add(b.subtract(a).multiply(f))
    loglin = a.max(1e-6).log().multiply(1 - f).add(b.max(1e-6).log().multiply(f)).exp()
    # log-linear onde as duas épocas são > 0; linear onde alguma é 0
    return lin.where(a.gt(0).And(b.gt(0)), loglin).rename("POP")


# ============================================================================
# 8) Vulnerabilidade V e verificação de degenerescência da justiça
# ============================================================================
def _census_img(y):
    spec = V_CENSUS[y]
    fc = ee.FeatureCollection(spec["asset"]).filter(ee.Filter.notNull([spec["prop"]]))
    return fc.reduceToImage([spec["prop"]], ee.Reducer.first())   # valor do setor, sem reamostrar

def vuln_raw(year):
    """Variável socioeconômica (maior = MENOS vulnerável) e proveniência."""
    if V_SOURCE == "census":
        yrs = sorted(int(k) for k in V_CENSUS)
        if not yrs:
            raise ValueError("V_SOURCE='census' exige V_CENSUS preenchido.")
        lo = max([y for y in yrs if y <= year], default=yrs[0])
        hi = min([y for y in yrs if y >= year], default=yrs[-1])
        a = _census_img(lo)
        if lo == hi:
            return a.rename("VAR"), ("observado" if lo == year else "mantido_%d" % lo)
        b = _census_img(hi)
        f = (year - lo) / float(hi - lo)
        x = a.max(1e-6).log().multiply(1 - f).add(b.max(1e-6).log().multiply(f)).exp()
        return x.rename("VAR"), "interpolado_loglinear"
    if not GDP_ASSET:
        raise ValueError("V_SOURCE='gdp_adm1' exige GDP_ASSET na configuração.")
    y = min(max(year, GDP_YEAR_START), GDP_YEAR_END)
    prov = "observado" if y == year else "mantido_%d" % y
    return ee.Image(GDP_ASSET).select(f"PPP_{y}").rename("VAR"), prov

def vulnerability(year, ctx):
    x, prov = vuln_raw(year)
    # percentis da cidade: só uma estatística (escala de amostragem); a camada não muda
    pct = x.reduceRegion(ee.Reducer.percentile([2, 98]), ctx["region"], V_STATS_SCALE,
                         maxPixels=1e12, bestEffort=True)
    p2  = ee.Number(ee.Algorithms.If(pct.get("VAR_p2"),  pct.get("VAR_p2"),  0))
    p98 = ee.Number(ee.Algorithms.If(pct.get("VAR_p98"), pct.get("VAR_p98"), 0))
    rng = p98.subtract(p2)
    # V constante na cidade (ex.: PIB estadual) => J não tem informação
    j_valid = rng.gt(p98.abs().multiply(V_MIN_REL_RANGE).max(1e-12))
    sc = x.subtract(p2).divide(rng.max(1e-12)).clamp(0, 1)
    v = ee.Image.constant(1).subtract(sc).rename("V")
    return v, ee.Number(j_valid), prov


# ============================================================================
# 9) Focais ponderados por população — na grade de SAÍDA (30 m), sem agregação
#    Janela quadrada de WIN_RADIUS_M (em metros). "boxcar" = soma móvel O(1) por pixel.
# ============================================================================
def _kernel():
    return ee.Kernel.square(radius=WIN_RADIUS_M, units="meters")

def _nsum(img, K):
    return img.reduceNeighborhood(reducer=ee.Reducer.sum(), kernel=K, optimization="boxcar")

def _wmean(img, w, K):
    valid = img.mask().gt(0)
    num = _nsum(img.unmask(0).multiply(w).multiply(valid), K)
    # denominador só com pixels onde img é válido
    den = _nsum(w.multiply(valid), K).max(1e-9)
    return num.divide(den)

def equity_local(eq, w):
    K = _kernel()
    mu = _wmean(eq, w, K)
    ex2 = _wmean(eq.pow(2), w, K)
    sd = ex2.subtract(mu.pow(2)).max(0).sqrt()
    cv = sd.divide(mu.max(1e-6)).min(1.0)
    return ee.Image.constant(1).subtract(cv).clamp(NI_FLOOR, 1.0).rename("E_local")

def justice_local(eq, w, v):
    K = _kernel()
    b = ee.Image.constant(1).subtract(eq)                 # ônus ambiental
    v = v.updateMask(eq.mask())
    mb, mv = _wmean(b, w, K), _wmean(v, w, K)
    cov = _wmean(b.multiply(v), w, K).subtract(mb.multiply(mv))
    sb = _wmean(b.pow(2), w, K).subtract(mb.pow(2)).max(0).sqrt()
    sv = _wmean(v.pow(2), w, K).subtract(mv.pow(2)).max(0).sqrt()
    corr = cov.divide(sb.multiply(sv).max(1e-6))          # + = ônus sobre os vulneráveis
    return ee.Image.constant(1).subtract(corr.clamp(0, 1)).clamp(NI_FLOOR, 1.0).rename("J_local")


# ============================================================================
# 10) Fusão  HU-EEQI = EQ^a · E^b · J^c  (expoentes efetivos no servidor)
# ============================================================================
def fuse_index(eq, e_loc, j_loc, j_valid):
    jv = ee.Number(j_valid) if USE_JUSTICE else ee.Number(0)
    c_eff = ee.Number(GAMMA).multiply(jv)
    if J_FALLBACK == "to_eq":
        a_eff = ee.Number(ALPHA).add(ee.Number(GAMMA).subtract(c_eff))
    else:                                   # "renormalize": alfa e beta proporcionais
        a_eff = ee.Number(ALPHA)
    s = a_eff.add(BETA).add(c_eff)
    ex = {"a": a_eff.divide(s), "b": ee.Number(BETA).divide(s), "c": c_eff.divide(s)}
    log = (eq.log().multiply(ex["a"])
           .add(e_loc.log().multiply(ex["b"]))
           .add(j_loc.log().multiply(ex["c"])))
    return ee.Image(log).exp().clamp(0, 1).rename("HU_EEQI"), ex


# ============================================================================
# 11) Montagem do ano/cidade em DUAS passagens — [6] bandas NI_*, n_obs, sensor
#     Nenhuma camada é reprojetada, agregada ou interpolada.
#     1ª passagem (cara, sem calibração): raw_year_image = descritores brutos + QA + POP + V,
#        gravada em RAW_COLLECTION (notebook 00). É a única que lê Landsat/ERA5/MAIAC.
#     Calibração (notebook 01): amostras tiradas dessas imagens já gravadas (rápido).
#     2ª passagem (barata): index_from_raw = normalização, EQ, E, J e HU-EEQI sobre a pilha
#        gravada (notebook 02).
#     build_year_image = as duas passagens em sequência, sem gravar (mesmo resultado; usado nos
#     testes e nas variantes do notebook 05).
# ============================================================================
RAW_QA_BANDS = ["n_obs", "sensor_flag", "n_TM", "n_ETM", "n_OLI", "n_obs_thermal", "LST", "LST_anom",
                "n_aod", "AQ_filled", "HEAT_ndays_xc", "HEAT_filled"]
RAW_BANDS = DESCRIPTORS + RAW_QA_BANDS + ["POP", "V"]

def raw_config():
    """Parâmetros que mudam a pilha bruta (os limites só entram na 2ª passagem)."""
    c = config_snapshot()
    c.pop("BOUNDS_PCT", None)
    c.update({"USE_JUSTICE": USE_JUSTICE, "V_SOURCE": V_SOURCE, "POP_SOURCE": POP_SOURCE,
              "WIN_RADIUS_M": WIN_RADIUS_M})
    return json.loads(json.dumps(c))

def raw_year_image(year, ctx, DOM):
    raw, qa = raw_stack(year, ctx)
    raw = raw.updateMask(DOM)
    pop = pop_year(year)
    if USE_JUSTICE:
        v, j_valid, v_prov = vulnerability(year, ctx)
    else:
        v, j_valid, v_prov = ee.Image.constant(0).rename("V"), ee.Number(0), "n/a"
    lst = qa.select("LST")
    med = lst.updateMask(DOM).reduceRegion(ee.Reducer.median(), ctx["region"], 90,
                                           maxPixels=1e12, bestEffort=True).get("LST")
    lst_anom = lst.subtract(ee.Number(ee.Algorithms.If(med, med, 0))).rename("LST_anom")
    bands = [raw, qa.select(["n_obs", "sensor_flag", "n_TM", "n_ETM", "n_OLI", "n_obs_thermal", "LST"]),
             lst_anom, qa.select(["n_aod", "AQ_filled", "HEAT_ndays_xc", "HEAT_filled"]), pop, v]
    img = ee.Image.cat([ee.Image(b).toFloat() for b in bands]).select(RAW_BANDS)
    # recorte no BUFFER (região + WIN_RADIUS_M + 1 km): a janela de E/J nas bordas do município usa
    # pixels de fora dele, como no cálculo em uma passagem
    return img.clip(ctx["region_buf"]).set({
        "year": year, "city": ctx["name"], "version": VERSION, "stage": "raw", "j_valid": j_valid,
        "prov": json.dumps({"POP": pop_provenance(year), "V": v_prov, "AQ": "observado",
                            "HEAT": "observado"}),
        "raw_config": json.dumps(raw_config()), "export_scale_m": EQ_SCALE,
        "note": ("Pilha bruta (1ª passagem): descritores sem normalização, QA, POP e V. O índice é "
                 "calculado depois (notebook 02) com a calibração tirada destas imagens (notebook 01).")})

def index_from_raw(r, year, ctx, bounds, weights, DOM):
    r = ee.Image(r)
    raw = r.select(DESCRIPTORS)
    ni, clamped = normalize_stack(raw, bounds)
    eq = compute_eq(ni, weights)
    pop, v = r.select("POP"), r.select("V")
    w = pop.unmask(0).add(POP_W_EPS)
    j_valid = ee.Number(r.get("j_valid"))
    e_loc = equity_local(eq, w)
    j_loc = justice_local(eq, w, v) if USE_JUSTICE else ee.Image.constant(1.0).rename("J_local")
    e_loc, j_loc = e_loc.updateMask(eq.mask()), j_loc.updateMask(eq.mask())
    hu, ex = fuse_index(eq, e_loc, j_loc, j_valid)
    bands = [hu, eq, e_loc, j_loc, ni, r.select(["n_obs", "sensor_flag"]), pop, v,
             urban_mask(year), DOM.rename("domain")]
    if EXPORT_DIAGNOSTICS:
        bands += [raw, clamped, r.select(["n_TM", "n_ETM", "n_OLI", "n_obs_thermal", "LST", "LST_anom",
                                          "n_aod", "AQ_filled", "HEAT_ndays_xc", "HEAT_filled"])]
    img = ee.Image.cat([ee.Image(b).toFloat() for b in bands])
    img = img.select(CORE_BANDS + (DIAG_BANDS if EXPORT_DIAGNOSTICS else []))
    props = {
        "year": year, "city": ctx["name"], "version": VERSION,
        "harmonize": str(HARMONIZE), "roy_method": ROY_METHOD, "l7_policy": L7_POLICY,
        "heat_method": HEAT_METHOD, "heat_stat": HEAT_STAT, "aq_source": AQ_SOURCE,
        "aq_fill": AQ_FILL, "pop_source": POP_SOURCE, "v_source": V_SOURCE,
        "j_fallback": J_FALLBACK, "annual_statistic": "median",
        "prov": r.get("prov"),
        "weights": json.dumps(weights), "bounds": json.dumps(bounds),
        "alpha": ALPHA, "beta": BETA, "gamma": GAMMA, "win_radius_m": WIN_RADIUS_M,
        "ni_floor": NI_FLOOR, "export_scale_m": EQ_SCALE,
        "j_valid": j_valid, "exp_eq": ex["a"], "exp_e": ex["b"], "exp_j": ex["c"],
        "note": ("Camadas na resolução original, sem reprojeção/agregação/interpolação: "
                 "Landsat 30 m, ERA5-Land ~9 km, MAIAC 1 km, GHS-POP 100 m, PIB adm1. "
                 "Agregação anual pela mediana. E/J com janela de WIN_RADIUS_M na grade de exportação. "
                 "Índice calculado sobre a pilha bruta gravada (2ª passagem)"),
    }
    return img.clip(ctx["region"]).set(props)

def build_year_image(year, ctx, bounds, weights, DOM):
    return index_from_raw(raw_year_image(year, ctx, DOM), year, ctx, bounds, weights, DOM)

def raw_asset_id_for(ctx, year):
    return f"{RAW_COLLECTION}/huEEQIraw_{ctx['safe']}_{year}"

def raw_export_kwargs(img, ctx, year):
    pyr = {".default": "mean", "sensor_flag": "mode", "AQ_filled": "mode", "HEAT_filled": "mode"}
    return dict(image=img, description=f"huEEQIraw{VERSION}_{ctx['safe']}_{year}"[:100],
                assetId=raw_asset_id_for(ctx, year), region=ctx["region_buf"],
                scale=EQ_SCALE, crs=EXPORT_CRS, maxPixels=1e13, pyramidingPolicy=pyr)

def check_raw_config(ids):
    """Para se alguma pilha bruta gravada foi feita com outra configuração."""
    if not ids:
        return
    cfgs = (ee.ImageCollection.fromImages([ee.Image(i) for i in ids])
            .aggregate_array("raw_config").distinct().getInfo())
    cur, bad = raw_config(), []
    for s in cfgs:
        old = json.loads(s)
        diff = {k: {"bruto": old.get(k), "atual": v} for k, v in cur.items() if old.get(k) != v}
        if diff:
            bad.append(diff)
    if bad:
        raise ValueError(f"Há pilhas brutas em {RAW_COLLECTION} feitas com outra configuração. Apague-as "
                         "(ou volte a configuração) e rode o notebook 00 de novo.\n"
                         + json.dumps(bad, ensure_ascii=False, indent=1))

def asset_id_for(ctx, year):
    return f"{OUT_COLLECTION}/huEEQI_{ctx['safe']}_{year}"

def export_kwargs(img, ctx, year):
    pyr = {".default": "mean", "sensor_flag": "mode", "NI_clamped": "mode", "AQ_filled": "mode",
           "HEAT_filled": "mode", "urban": "mode", "domain": "mode"}
    return dict(image=img, description=f"huEEQIv6_{ctx['safe']}_{year}"[:100],
                assetId=asset_id_for(ctx, year), region=ctx["region"],
                scale=EQ_SCALE, crs=EXPORT_CRS, maxPixels=1e13, pyramidingPolicy=pyr)

def existing_assets(parent):
    ids, token = set(), None
    try:
        while True:
            params = {"parent": parent}
            if token:
                params["pageToken"] = token
            r = ee.data.listAssets(params)
            ids |= {a.get("id") or a.get("name") for a in r.get("assets", [])}
            token = r.get("nextPageToken")
            if not token:
                break
    except Exception as e:
        print("Aviso: não listei assets existentes:", str(e)[:120])
    return ids

def monitor(prefix, max_ops=2000):
    ops = [o for o in ee.data.listOperations()[:max_ops]
           if o.get("metadata", {}).get("description", "").startswith(prefix)]
    states = {}
    for o in ops:
        s = o["metadata"].get("state")
        states[s] = states.get(s, 0) + 1
    print(prefix, states)
    for o in ops:
        if o["metadata"].get("state") == "FAILED":
            print("  FALHOU:", o["metadata"].get("description"), "->",
                  o.get("error", {}).get("message", "")[:200])
    return states


# ============================================================================
# 13) Destinos e [9] preenchimento temporal de lacunas (notebook 02b; usado pelos 04 e 05)
# ============================================================================
def ensure_collection(path, type_="IMAGE_COLLECTION"):
    """Cria a coleção/pasta se não existir. Sem permissão: para AQUI, antes de criar tasks
    (uma task para destino inexistente só falha depois, com "Asset ... not found")."""
    try:
        ee.data.getAsset(path)
        return
    except Exception:
        pass
    try:
        ee.data.createAsset({"type": type_}, path)
        print("Criado:", path)
    except Exception as e:
        raise PermissionError(f"Não consegui criar {path}. A conta autenticada precisa de permissão "
                              "de escrita no projeto/pasta desse caminho. Nenhuma task foi criada.\n"
                              f"Detalhe: {str(e)[:300]}")

def available_bands():
    return CORE_BANDS + (DIAG_BANDS if EXPORT_DIAGNOSTICS else [])

def fill_band_list():
    """Bandas contínuas preenchidas pela cascata (FILL_BANDS presentes no índice)."""
    allb = available_bands()
    return [b for b in FILL_BANDS if b in allb]

def fill_copy_bands():
    """Contagens e marcas copiadas do ano observado (0 = sem observação/sem marca naquele ano)."""
    allb = available_bands()
    return [b for b in FILL_COPY_BANDS if b in allb]

def _masked(bands):
    return ee.Image.constant([0] * len(bands)).rename(bands).toFloat().updateMask(0)

def _value_and_year(img, year, FB):
    """Bandas FB + uma banda de ano por banda, mascarada igual ao valor."""
    v = ee.Image(img).select(FB).toFloat()
    t = (ee.Image.constant([float(year)] * len(FB))
         .rename([b + "__t" for b in FB]).toFloat())
    return v.addBands(t.updateMask(v.mask()))

def _first_valid(series, years, FB):
    """Para cada pixel e banda, o 1º valor válido percorrendo `years` na ordem dada, e o ano dele."""
    if not years:
        return _masked(FB), _masked(FB)
    tb = [b + "__t" for b in FB]
    r = (ee.ImageCollection.fromImages([_value_and_year(series[y], y, FB) for y in years])
         .reduce(ee.Reducer.firstNonNull()).rename(FB + tb))
    return r.select(FB), r.select(tb).rename(FB)

def city_series(col, ctx):
    """{ano: imagem observada} da capital (1 getInfo para saber os anos presentes)."""
    cc = col.filter(ee.Filter.eq("city", ctx["name"])).filter(ee.Filter.inList("year", YEARS))
    years = sorted(set(int(y) for y in cc.aggregate_array("year").getInfo()))
    return {y: ee.Image(cc.filter(ee.Filter.eq("year", y)).first()) for y in years}

def _spatial_mean(img, FB):
    """Tipo 3: média, banda a banda, dos pixels COM valor numa janela quadrada de WIN_RADIUS_M
    (a mesma janela de E e J)."""
    K = _kernel()
    num, den = _nsum(img.unmask(0), K), _nsum(img.mask(), K)
    return num.divide(den.max(1e-9)).updateMask(den.gt(0)).rename(FB)

def fill_year_image(series, year, ctx, DOM):
    """Imagem do `year` sem lacunas no domínio, pela cascata (pixel a pixel, banda a banda), em TODAS
    as bandas contínuas (fill_band_list(): índice, componentes, descritores, LST, POP e V):
      0 observado;
      1 interpolação linear no tempo entre o último ano com dado ANTES e o primeiro DEPOIS
        (lacuna de 1 ano = média dos dois; vale também para o ano inteiro vazio);
      2 ponta da série (só um lado tem dado): repete o ano com dado mais próximo;
      3 sem dado em nenhum ano: média dos pixels com valor no mesmo ano numa janela de WIN_RADIUS_M;
      4 nenhum pixel com valor na janela: mediana da capital no ano (pixels observados; se o ano não
        tiver nenhum, mediana da camada preenchida no tempo).
    fill_type e gap_years descrevem FILL_KEY_BAND; filled_mask marca (bit i) as bandas FB[i] não
    observadas no pixel-ano. series = {ano: ee.Image observada}."""
    FB = fill_band_list()
    present = sorted(series)
    cur = (ee.Image(series[year]).select(FB).toFloat() if year in series else _masked(FB))
    pv, pt = _first_valid(series, [y for y in reversed(present) if y < year], FB)   # antes
    nv, nt = _first_valid(series, [y for y in present if y > year], FB)             # depois
    yr = ee.Image.constant(float(year))
    interp = pv.add(nv.subtract(pv).multiply(yr.subtract(pt).divide(nt.subtract(pt))))   # tipo 1
    edge = pv.unmask(nv)                                                                  # tipo 2
    temporal = cur.unmask(interp).unmask(edge)                                            # tipos 0–2
    spatial = _spatial_mean(temporal.updateMask(DOM), FB)                                 # tipo 3
    tmp = [b + "__tmp" for b in FB]
    med = (cur.addBands(temporal.rename(tmp)).updateMask(DOM)
           .reduceRegion(ee.Reducer.median(), ctx["region"], EQ_SCALE, crs=EXPORT_CRS,
                         maxPixels=1e13, tileScale=4))
    consts = [ee.Image.constant(ee.Number(ee.Algorithms.If(
                  med.get(b), med.get(b), ee.Algorithms.If(med.get(t), med.get(t), -9999))))
              for b, t in zip(FB, tmp)]
    city_med = ee.Image.cat(consts).rename(FB).toFloat()
    city_med = city_med.updateMask(city_med.neq(-9999))
    filled = temporal.unmask(spatial).unmask(city_med)                                    # tipo 4

    k = FILL_KEY_BAND
    ftype = (ee.Image.constant(4).where(spatial.select(k).mask(), 3).where(edge.select(k).mask(), 2)
             .where(interp.select(k).mask(), 1).where(cur.select(k).mask(), 0))
    kpt, knt = pt.select(k), nt.select(k)
    gap = (knt.subtract(kpt)                    # 1: distância entre os dois anos observados
           .unmask(yr.subtract(kpt))            # 2: só há ano antes
           .unmask(knt.subtract(yr))            # 2: só há ano depois
           .unmask(float(len(YEARS)))           # 3 e 4: nunca observado
           .where(cur.select(k).mask(), 0))
    fmask = ee.Image.constant(0)
    for i, b in enumerate(FB):
        fmask = fmask.add(cur.select(b).mask().Not().multiply(2 ** i))
    parts = [filled, ftype.rename("fill_type").toFloat(), gap.rename("gap_years").toFloat(),
             fmask.rename("filled_mask").toFloat()]
    cb = fill_copy_bands()
    if cb:
        parts.append(ee.Image(series[year]).select(cb).unmask(0).toFloat() if year in series
                     else ee.Image.constant([0] * len(cb)).rename(cb).toFloat())
    if "urban" in available_bands():
        parts.append(urban_mask(year).toFloat())
    img = ee.Image.cat(parts).updateMask(DOM).addBands(DOM.rename("domain").toFloat()).clip(ctx["region"])
    props = {"year": year, "city": ctx["name"], "version": VERSION, "source": OUT_COLLECTION,
             "year_observed": int(year in series), "years_present": json.dumps(present),
             "fill_key_band": k, "fill_bands": json.dumps(FB),
             "fill_method": ("0 observado | 1 interpolação linear no tempo entre o último ano com dado antes "
                             "e o primeiro depois | 2 ponta da série: ano com dado mais próximo | 3 nunca "
                             f"observado: média dos pixels com valor no mesmo ano numa janela de {WIN_RADIUS_M} m"
                             " | 4 sem valor na janela: mediana da capital no ano"),
             "note": ("Preenchimento pixel a pixel sobre a série anual exportada (02b). filled_mask: bit i = "
                      "banda fill_bands[i] preenchida. NI_clamped, AQ_filled e HEAT_filled = 0 onde o ano "
                      "não foi observado.")}
    img = img.set(props)
    if year in series:
        img = ee.Image(img.copyProperties(series[year], ["j_valid", "exp_eq", "exp_e", "exp_j", "prov"]))
    return img

def filled_asset_id(ctx, year):
    return f"{FILLED_COLLECTION}/huEEQIf_{ctx['safe']}_{year}"

def fill_export_kwargs(img, ctx, year):
    pyr = {".default": "mean", "fill_type": "mode", "gap_years": "mode", "filled_mask": "mode",
           "sensor_flag": "mode", "NI_clamped": "mode", "AQ_filled": "mode", "HEAT_filled": "mode",
           "urban": "mode", "domain": "mode"}
    return dict(image=img, description=f"huEEQIf{VERSION}_{ctx['safe']}_{year}"[:100],
                assetId=filled_asset_id(ctx, year), region=ctx["region"], scale=EQ_SCALE,
                crs=EXPORT_CRS, maxPixels=1e13, pyramidingPolicy=pyr)

FILL_SUMMARY_COLS = ["n_domain", "n_observed", "n_interp", "n_edge", "n_spatial", "n_city_median",
                     f"n_gap_gt{FILL_REPORT_GAP}", "n_still_missing", "n_any_band_missing"]

def fill_summary_feature(img, ctx):
    """Pixels do domínio por tipo de preenchimento, lacunas > FILL_REPORT_GAP anos e pixels que ainda
    têm alguma banda sem dado (n_still_missing e n_any_band_missing devem ser 0)."""
    ft, gap = img.select("fill_type"), img.select("gap_years")
    still = ft.gte(0).And(img.select(FILL_KEY_BAND).mask().Not())
    any_miss = ft.gte(0).And(img.select(fill_band_list()).mask().reduce(ee.Reducer.min()).eq(0))
    ind = ee.Image.cat([ft.gte(0), ft.eq(0), ft.eq(1), ft.eq(2), ft.eq(3), ft.eq(4),
                        ft.gt(0).And(ft.lt(3)).And(gap.gt(FILL_REPORT_GAP)), still, any_miss]
                       ).rename(FILL_SUMMARY_COLS)
    s = ind.reduceRegion(ee.Reducer.sum().unweighted(), ctx["region"], EQ_SCALE, crs=EXPORT_CRS,
                         maxPixels=1e13, tileScale=4)
    return ee.Feature(None, s).set({"city": ctx["name"], "year": img.get("year")})

def fill_series_np(X, years, groups=None):
    """Referência em numpy da cascata TEMPORAL (linhas = pixels, colunas = anos).
    Devolve (valores preenchidos, fill_type, gap_years). Tipos 0–2 iguais ao GEE. Pixel nunca
    observado: num painel de pontos não há vizinhos espaciais (tipo 3 do GEE), então recebe a
    mediana por ano dos valores OBSERVADOS das linhas do mesmo grupo (cidade) -> tipo 4."""
    import numpy as np, warnings
    X = np.asarray(X, float)
    m, n = X.shape
    t = np.asarray(years, float)
    obs = ~np.isnan(X)
    Xf, typ = X.copy(), np.where(obs, 0.0, 4.0)
    gap = np.where(obs, 0.0, float(len(YEARS)))
    for r in range(m):
        idx = np.nonzero(obs[r])[0]
        if not len(idx):
            continue
        for c in np.nonzero(~obs[r])[0]:
            b, a = idx[idx < c], idx[idx > c]
            if len(b) and len(a):
                b, a = b[-1], a[0]
                Xf[r, c] = X[r, b] + (X[r, a] - X[r, b]) * (t[c] - t[b]) / (t[a] - t[b])
                typ[r, c], gap[r, c] = 1, t[a] - t[b]
            elif len(b):
                Xf[r, c], typ[r, c], gap[r, c] = X[r, b[-1]], 2, t[c] - t[b[-1]]
            else:
                Xf[r, c], typ[r, c], gap[r, c] = X[r, a[0]], 2, t[a[0]] - t[c]
    never = ~obs.any(axis=1)
    if never.any():
        g = (np.zeros(m, int) if groups is None
             else np.unique(np.asarray(groups).astype(str), return_inverse=True)[1])
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            for gi in np.unique(g[never]):
                rows = g == gi
                med = np.nanmedian(X[rows], axis=0)                       # observados do ano
                seen = rows & ~never
                if seen.any():
                    med = np.where(np.isnan(med), np.nanmedian(Xf[seen], axis=0), med)
                Xf[rows & never] = med
    return Xf, typ, gap


In [ ]:
# @title G1. Janela sazonal fixa por capital (climatologia ERA5-Land, mesmo período da série)
# Critério: a estação meteorológica padrão de 3 meses (DJF, MAM, JJA, SON) com menor
# precipitação média em YEAR_START–YEAR_END, a partir de ERA5-Land MONTHLY_AGGR
# (total_precipitation_sum), a mesma família de dados ERA5-Land já usada na v3.
import pandas as pd
SEASONS = {"DJF": (12, 2), "MAM": (3, 5), "JJA": (6, 8), "SON": (9, 11)}
SEASON_MONTHS_OF = {"DJF": [12, 1, 2], "MAM": [3, 4, 5], "JJA": [6, 7, 8], "SON": [9, 10, 11]}
era = (ee.ImageCollection("ECMWF/ERA5_LAND/MONTHLY_AGGR")
       .filterDate(f"{YEAR_START}-01-01", f"{YEAR_END + 1}-01-01")
       .select("total_precipitation_sum"))
clim = ee.Image.cat([era.filter(ee.Filter.calendarRange(m, m, "month")).mean()
                     .multiply(1000).rename(f"m{m:02d}") for m in range(1, 13)])   # mm/mês

CTXS = city_contexts()
rows = []
for ctx in CTXS:
    v = clim.reduceRegion(ee.Reducer.mean(), ctx["region"], 11132, maxPixels=1e10,
                          bestEffort=True).getInfo()
    mm = {m: v.get(f"m{m:02d}") for m in range(1, 13)}
    tot = {s: sum(mm[m] for m in ms) for s, ms in SEASON_MONTHS_OF.items()}
    dry = min(tot, key=tot.get)
    rows.append({"city": ctx["name"], **{f"P_{m:02d}_mm": round(mm[m], 1) for m in mm},
                 **{f"P_{s}_mm": round(t, 1) for s, t in tot.items()}, "season": dry,
                 "months": SEASONS[dry]})
season_tab = pd.DataFrame(rows)
OUT_DIR = DRIVE_DIR + "/robustez_05"
os.makedirs(OUT_DIR, exist_ok=True)
season_tab.to_csv(f"{OUT_DIR}/season_by_city.csv", index=False)
SEASON_MONTHS = {r["city"]: tuple(r["months"]) for r in rows}
print(season_tab[["city", "season"] + [f"P_{s}_mm" for s in SEASONS]].to_string(index=False))

In [ ]:
# @title G2. Exportar o painel: base (assets v6), tendências GEE e variantes (nos mesmos pontos)
with open(CALIB_JSON) as f:
    CALIB = json.load(f)
BOUNDS, WEIGHTS = CALIB["bounds"], CALIB["weights_all"][WEIGHT_METHOD]
DOM = domain_mask()
SERIES_COLLECTION = FILLED_COLLECTION if TREND_SOURCE == "filled" else OUT_COLLECTION
col = ee.ImageCollection(SERIES_COLLECTION).filter(ee.Filter.inList("year", YEARS))
BASE_BANDS = (["HU_EEQI", "n_obs", "n_TM", "n_ETM", "n_OLI", "sensor_flag"]
              + (["fill_type", "gap_years"] if TREND_SOURCE == "filled" else []))
PT_PROPS = ["pid"]
pt_img = DOM.selfMask().rename("dom")
if MAPBIOMAS_ASSET:
    mb = ee.Image(MAPBIOMAS_ASSET)
    mb_bands = [f"classification_{y}" for y in range(MB_YEARS[0], MB_YEARS[1] + 1)]
    pt_img = pt_img.addBands(mb.select(mb_bands).reduce(ee.Reducer.countDistinct())
                             .eq(1).rename("mb_stable"))
    PT_PROPS.append("mb_stable")
else:
    print("HU_MAPBIOMAS_ASSET não definido: a comparação em áreas estáveis será omitida.")

def panel_points(ctx):
    """Mesmos pontos do notebook 03: estratificado no domínio, semente PANEL_SEED + i."""
    def setpid(f):
        return f.set("pid", ee.String(str(ctx["idx"]) + "_").cat(ee.String(f.get("system:index"))))
    return (pt_img.stratifiedSample(numPoints=N_PANEL_PER_CITY, classBand="dom",
                                    region=ctx["region"], scale=EQ_SCALE,
                                    seed=PANEL_SEED + ctx["idx"], geometries=True, tileScale=4)
            .map(setpid))

def sampler(pts, bands, extra=None):
    def f(im):
        props = {"year": im.get("year"), "city": im.get("city")}
        return (im.select(bands).sampleRegions(collection=pts, properties=PT_PROPS,
                                               scale=EQ_SCALE, geometries=True, tileScale=4)
                .map(lambda ft: ft.set(props)))
    return f

def start(fc, name, selectors):
    t = ee.batch.Export.table.toDrive(collection=fc, description=name[:100], folder=DRIVE_FOLDER,
                                      fileNamePrefix=name, fileFormat="CSV", selectors=selectors)
    t.start()
    return t

SEL = ["city", "year"] + PT_PROPS + BASE_BANDS + [".geo"]
GEE_VALID_BANDS = ["HU_EEQI_" + b for b in
                   ["S", "tau", "sen_slope", "p_value", "p_value_ties", "p_value_hr",
                    "p_value_tfpw", "r1_resid", "class_mk", "class_mk_ties", "class_hr",
                    "class_tfpw", "complete"]]
base_parts, gee_parts = [], []
trends_ready = existing_assets(TRENDS_COLLECTION) if TRENDS_COLLECTION else set()
for ctx in CTXS:
    pts = panel_points(ctx)
    cc = col.filter(ee.Filter.eq("city", ctx["name"]))
    base_parts.append(ee.FeatureCollection(cc.map(sampler(pts, BASE_BANDS))).flatten())
    tr_id = f"{TRENDS_COLLECTION}/trend_{ctx['safe']}_{YEAR_START}_{YEAR_END}"
    if tr_id not in trends_ready:
        if TRENDS_COLLECTION:
            print("  sem tendência exportada (notebook 04) para", ctx["name"], "- fora da validação GEE")
        continue
    if TRENDS_COLLECTION:
        tr = ee.Image(tr_id)
        # selecionar ANTES de amostrar: tau_p é mascarada onde não há significância e o
        # sampleRegions descartaria esses pixels (validação ficaria enviesada)
        gee_parts.append(tr.select(GEE_VALID_BANDS)
                           .sampleRegions(collection=pts, properties=PT_PROPS, scale=EQ_SCALE,
                                          geometries=False, tileScale=4)
                           .map(_setter({"city": ctx["name"]})))
start(ee.FeatureCollection(base_parts).flatten(), "rob_base", SEL)
if gee_parts:
    start(ee.FeatureCollection(gee_parts).flatten(), "rob_gee_trends",
          ["city"] + PT_PROPS + GEE_VALID_BANDS)

# ---- variantes: índice recalculado nos pontos com a MESMA calibração da base ----------
VARIANTS = {"noharm": {"HARMONIZE": False}, "season": {"_season": True}}
VAR_BANDS = ["HU_EEQI", "n_obs", "n_TM", "n_ETM", "n_OLI", "sensor_flag"]

def build_variant(y, ctx, spec):
    saved = {k: globals()[k] for k in spec if not k.startswith("_")}
    try:
        for k, v in spec.items():
            if not k.startswith("_"):
                globals()[k] = v
        c = dict(ctx, months=SEASON_MONTHS[ctx["name"]]) if spec.get("_season") else ctx
        return build_year_image(y, c, BOUNDS, WEIGHTS, DOM).select(VAR_BANDS)
    finally:
        globals().update(saved)

n_tasks = 0
for vname, spec in VARIANTS.items():
    for ctx in CTXS:
        pts = panel_points(ctx)
        imgs = ee.ImageCollection.fromImages([build_variant(y, ctx, spec) for y in YEARS])
        fc = ee.FeatureCollection(imgs.map(sampler(pts, VAR_BANDS))).flatten()
        start(fc, f"rob_{vname}_{ctx['safe']}", ["city", "year"] + PT_PROPS + VAR_BANDS + [".geo"])
        n_tasks += 1
print("Exportações: rob_base,", "rob_gee_trends," if gee_parts else "", n_tasks, "variantes")

In [ ]:
# @title Acompanhar
monitor("rob_")

In [ ]:
# @title Funções: MK (a–d), Sen, ACF, FDR, Pettitt, kappa (numpy, vetorizado)
import math, glob, sys, platform
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from statistics import NormalDist
Z_CRIT = NormalDist().inv_cdf(1 - ALPHA_SIG / 2)
TESTS = ["mk", "mk_ties", "hr", "tfpw", "hr_fdr"]
TEST_LABEL = {"mk": "MK original", "mk_ties": "MK ties-corrected", "hr": "Hamed-Rao MK",
              "tfpw": "TFPW MK", "hr_fdr": "Hamed-Rao MK + BH-FDR"}
PRINCIPAL = "hr"

def _pairs(n):
    return np.triu_indices(n, 1)

def mk_S(X):
    i, j = _pairs(X.shape[1])
    return np.sign(X[:, j] - X[:, i]).sum(1)

def ties_term(X):
    c = (X[:, :, None] == X[:, None, :]).sum(2)          # c_i = nº de valores iguais a x_i
    return ((c - 1) * (2 * c + 5)).sum(1)                # = Σ_grupos t(t-1)(2t+5)

def var_s(n, T=0.0):
    return (n * (n - 1) * (2 * n + 5) - T) / 18.0

def z_score(S, V):
    with np.errstate(all="ignore"):
        sd = np.sqrt(V)
        return np.where(S > 0, (S - 1) / sd, np.where(S < 0, (S + 1) / sd, 0.0))

_erfc = np.vectorize(math.erfc, otypes=[float])
def p_two_sided(z):
    z = np.asarray(z, float)
    out = np.full(z.shape, np.nan)
    ok = np.isfinite(z)
    out[ok] = _erfc(np.abs(z[ok]) / math.sqrt(2))
    return out

def sen_slope(X, t):
    i, j = _pairs(len(t))
    return np.median((X[:, j] - X[:, i]) / (t[j] - t[i]), axis=1)

def acf_biased(Y, nlags):
    """Igual ao pymannkendall: autocovariância com denominador n, normalizada pelo lag 0."""
    n = Y.shape[1]
    y = Y - Y.mean(1, keepdims=True)
    den = (y ** 2).sum(1)
    out = np.ones((Y.shape[0], nlags + 1))
    with np.errstate(all="ignore"):
        for k in range(1, nlags + 1):
            out[:, k] = (y[:, : n - k] * y[:, k:]).sum(1) / den
    return out

def avg_ranks(R):
    less = (R[:, None, :] < R[:, :, None]).sum(2)
    eq = (R[:, None, :] == R[:, :, None]).sum(2)
    return less + (eq + 1) / 2.0

def mk_family(X, t):
    """X: (m, n) séries completas e igualmente espaçadas; t: anos. Testes (a)–(d)."""
    m, n = X.shape
    idx = np.arange(1, n + 1, dtype=float)
    S, T = mk_S(X), ties_term(X)
    V0, Vt = var_s(n), var_s(n, T)
    beta = sen_slope(X, t)
    icpt = np.median(X - beta[:, None] * t[None, :], axis=1)
    # (c) Hamed & Rao (1998)
    rho = acf_biased(avg_ranks(X - beta[:, None] * idx[None, :]), n - 1)
    k = np.arange(1, n)
    w = (n - k) * (n - k - 1) * (n - k - 2)
    sig = np.abs(rho[:, 1:]) > Z_CRIT / math.sqrt(n)
    nns = 1 + 2.0 / (n * (n - 1) * (n - 2)) * (w * rho[:, 1:] * sig).sum(1)
    Vh = np.where(nns > 0, Vt * nns, np.nan)
    # (d) TFPW (Yue & Wang, 2002)
    X1 = X - beta[:, None] * idx[None, :]
    r1 = acf_biased(X1, 1)[:, 1]
    X3 = X1[:, 1:] - r1[:, None] * X1[:, :-1] + beta[:, None] * idx[None, : n - 1]
    S_d = mk_S(X3)
    Vd = var_s(n - 1, ties_term(X3))
    out = pd.DataFrame({"S": S, "tau": S / (n * (n - 1) / 2.0), "sen_slope": beta,
                        "sen_intercept": icpt, "r1_resid": r1, "n_ns_hr": nns, "S_tfpw": S_d})
    for name, z, s in [("mk", z_score(S, V0), S), ("mk_ties", z_score(S, Vt), S),
                       ("hr", z_score(S, Vh), S), ("tfpw", z_score(S_d, Vd), S_d)]:
        p = p_two_sided(z)
        out["z_" + name], out["p_" + name] = z, p
        out["class_" + name] = np.where(np.isnan(p), np.nan, np.where(p < ALPHA_SIG, np.sign(s), 0))
    return out

def bh_reject(p, q):
    p = np.asarray(p, float)
    rej = np.zeros(p.shape, bool)
    ok = np.isfinite(p)
    m = int(ok.sum())
    if m:
        ps = np.sort(p[ok])
        below = np.nonzero(ps <= q * np.arange(1, m + 1) / m)[0]
        if below.size:
            rej[ok] = p[ok] <= ps[below.max()]
    return rej

def pettitt(t, x):
    t, x = np.asarray(t), np.asarray(x, float)
    ok = ~np.isnan(x)
    t, x = t[ok], x[ok]
    n = len(x)
    if n < 6:
        return np.nan, np.nan
    sgn = np.sign(x[None, :] - x[:, None])
    U = np.array([sgn[: k + 1, k + 1:].sum() for k in range(n - 1)])
    k = int(np.argmax(np.abs(U)))
    p = min(1.0, 2 * math.exp(-6.0 * U[k] ** 2 / (n ** 3 + n ** 2)))
    return int(t[k + 1]), p                         # primeiro ano do novo regime

def cohen_kappa(a, b, labels=(-1, 0, 1)):
    a, b = np.asarray(a, float), np.asarray(b, float)
    ok = ~(np.isnan(a) | np.isnan(b))
    a, b = a[ok], b[ok]
    if not len(a):
        return np.nan, np.nan, 0
    po = float(np.mean(a == b))
    pe = float(sum(np.mean(a == l) * np.mean(b == l) for l in labels))
    return po, ((po - pe) / (1 - pe) if pe < 1 else np.nan), int(len(a))

def pct(x):
    x = np.asarray(x, float)
    x = x[~np.isnan(x)]
    return 100.0 * x.mean() if len(x) else np.nan

In [ ]:
# @title 1. Carregar painéis e rodar os testes (a)–(d) + FDR em cada versão
OUT_DIR = DRIVE_DIR + "/robustez_05"
os.makedirs(OUT_DIR, exist_ok=True)
T_YEARS = np.array(YEARS, dtype=float)

def load_panel(prefix):
    files = sorted(glob.glob(f"{DRIVE_DIR}/{prefix}*.csv"))
    if not files:
        return None
    d = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    if ".geo" in d:
        xy = d[".geo"].map(lambda g: json.loads(g)["coordinates"])
        d["lon"], d["lat"] = xy.str[0], xy.str[1]
    return d

def _pivot(d, band, P=None):
    t = d.pivot_table(index="pid", columns="year", values=band, aggfunc="first")
    return t.reindex(index=None if P is None else P.index, columns=YEARS)

def run_version(d):
    """Base: painel da série preenchida (02b), já sem lacunas e com fill_type/gap_years.
    Variantes (recalculadas nos pontos): mesma cascata em numpy quando TREND_SOURCE = "filled";
    com "observed", só séries completas (como na v4)."""
    P = _pivot(d, "HU_EEQI")
    meta_cols = [c for c in ["city", "lon", "lat", "mb_stable"] if c in d]
    meta = d.drop_duplicates("pid").set_index("pid")[meta_cols].loc[P.index]
    if "fill_type" in d and d["fill_type"].notna().any():     # série preenchida no GEE (02b)
        F, G = _pivot(d, "fill_type", P), _pivot(d, "gap_years", P)
    elif TREND_SOURCE == "filled":
        Xf, Ft, Gt = fill_series_np(P.to_numpy(float), YEARS, meta["city"].to_numpy())
        P = pd.DataFrame(Xf, index=P.index, columns=YEARS)
        F = pd.DataFrame(Ft, index=P.index, columns=YEARS)
        G = pd.DataFrame(Gt, index=P.index, columns=YEARS)
    else:
        F = pd.DataFrame(np.where(P.notna(), 0.0, np.nan), index=P.index, columns=YEARS)
        G = F.fillna(0)
    complete = P.notna().all(axis=1)
    res = mk_family(P[complete].to_numpy(float), T_YEARS)
    res.index = P.index[complete]
    res = meta.loc[res.index].join(res)
    res["n_filled"] = (F.loc[res.index] > 0).sum(axis=1)
    res["max_gap"] = G.loc[res.index].where(F.loc[res.index] > 0).max(axis=1).fillna(0)
    res["class_hr_fdr"] = res["class_hr"]
    for c, g in res.groupby("city"):
        rej = bh_reject(g["p_hr"].to_numpy(), FDR_Q)
        res.loc[g.index, "class_hr_fdr"] = np.where(g["class_hr"].isna(), np.nan,
                                                    np.where(rej, np.sign(g["S"]), 0))
    info = pd.DataFrame({"n_series": meta.groupby("city").size(),
                         "n_complete": complete.groupby(meta["city"]).sum(),
                         "n_nofill": (res["n_filled"] == 0).groupby(res["city"]).sum()})
    return res, info, P

PANELS = {"base": load_panel("rob_base"), "noharm": load_panel("rob_noharm_"),
          "season": load_panel("rob_season_")}
PANELS = {k: v for k, v in PANELS.items() if v is not None}
RES, INFO, SERIES = {}, {}, {}
for v, d in PANELS.items():
    RES[v], INFO[v], SERIES[v] = run_version(d)
    RES[v].to_csv(f"{OUT_DIR}/results_{v}.csv")
    print(f"{v}: {len(RES[v])} séries completas de {INFO[v].n_series.sum()} "
          f"({100 * len(RES[v]) / INFO[v].n_series.sum():.1f}%)")
CITIES = sorted(RES["base"]["city"].unique())

# Sensibilidade ao preenchimento: a base restrita aos pixels SEM nenhum ano preenchido
SUBSETS = []
if TREND_SOURCE == "filled" and "base" in RES:
    keep = RES["base"].index[RES["base"]["n_filled"] == 0]
    rs = RES["base"].loc[keep].copy()
    for c, g in rs.groupby("city"):                  # BH refeito dentro do subconjunto
        rej = bh_reject(g["p_hr"].to_numpy(), FDR_Q)
        rs.loc[g.index, "class_hr_fdr"] = np.where(g["class_hr"].isna(), np.nan,
                                                   np.where(rej, np.sign(g["S"]), 0))
    RES["base_nofill"], SERIES["base_nofill"] = rs, SERIES["base"].loc[keep]
    PANELS["base_nofill"] = PANELS["base"][PANELS["base"]["pid"].isin(keep)]
    INFO["base_nofill"] = pd.DataFrame({"n_series": rs.groupby("city").size(),
                                        "n_complete": rs.groupby("city").size(),
                                        "n_nofill": rs.groupby("city").size()})
    rs.to_csv(f"{OUT_DIR}/results_base_nofill.csv")
    SUBSETS.append("base_nofill")
    print(f"base_nofill: {len(rs)} séries sem nenhum ano preenchido "
          f"({100 * len(rs) / max(len(RES['base']), 1):.1f}% da base)")
VARIANT_KEYS = [v for v in RES if v != "base" and v not in SUBSETS]


In [ ]:
# @title 2. Validação: numpy vs pymannkendall e vs mapas GEE (notebook 04)
_pip("pymannkendall")
import pymannkendall as pmk
import ee as _ee_mod
VERS = {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
        "pymannkendall": pmk.__version__, "earthengine-api": _ee_mod.__version__}
print(VERS)

Pb = SERIES["base"].loc[RES["base"].index].to_numpy(float)
ref = {"mk_ties": pmk.original_test, "hr": pmk.hamed_rao_modification_test,
       "tfpw": pmk.trend_free_pre_whitening_modification_test}
rows = []
for name, fn in ref.items():
    p_ref = np.array([fn(x, alpha=ALPHA_SIG).p for x in Pb])
    p_np = RES["base"]["p_" + name].to_numpy()
    ok = np.isfinite(p_ref) & np.isfinite(p_np)
    rows.append({"test": name, "n": int(ok.sum()),
                 "max_abs_diff_p": float(np.max(np.abs(p_ref[ok] - p_np[ok]))) if ok.any() else np.nan,
                 "class_agreement_pct": pct((p_ref[ok] < ALPHA_SIG) == (p_np[ok] < ALPHA_SIG))})
slope_ref = np.array([pmk.sens_slope(x).slope for x in Pb])
rows.append({"test": "sen_slope", "n": len(Pb),
             "max_abs_diff_p": float(np.max(np.abs(slope_ref - RES["base"]["sen_slope"].to_numpy()))),
             "class_agreement_pct": np.nan})
gee = load_panel("rob_gee_trends")
if gee is not None:
    g = gee.set_index("pid").join(RES["base"], how="inner", rsuffix="_py")
    for gb, pb in [("HU_EEQI_p_value", "p_mk"), ("HU_EEQI_p_value_ties", "p_mk_ties"),
                   ("HU_EEQI_p_value_hr", "p_hr"), ("HU_EEQI_p_value_tfpw", "p_tfpw"),
                   ("HU_EEQI_sen_slope", "sen_slope")]:
        ok = g[gb].notna() & g[pb].notna()
        rows.append({"test": f"GEE {gb} vs numpy", "n": int(ok.sum()),
                     "max_abs_diff_p": float((g.loc[ok, gb] - g.loc[ok, pb]).abs().max()),
                     "class_agreement_pct": np.nan})
# preenchimento: GEE (02b) vs referência numpy nos pontos do painel (tipos 1 e 2)
db = PANELS["base"]
if "fill_type" in db:
    Pf = _pivot(db, "HU_EEQI")
    Ft = _pivot(db, "fill_type", Pf)
    Xr, Tr, _ = fill_series_np(Pf.where(Ft == 0).to_numpy(float), YEARS)
    m12 = np.isin(Ft.to_numpy(), [1, 2])
    rows.append({"test": "fill GEE (02b) vs numpy, tipos 1-2", "n": int(m12.sum()),
                 "max_abs_diff_p": float(np.abs(Xr[m12] - Pf.to_numpy(float)[m12]).max()) if m12.any() else np.nan,
                 "class_agreement_pct": pct(Tr[m12] == Ft.to_numpy()[m12]) if m12.any() else np.nan})
VALIDATION = pd.DataFrame(rows)
VALIDATION.to_csv(f"{OUT_DIR}/validation_pymannkendall.csv", index=False)
print(VALIDATION.to_string(index=False))

In [ ]:
# @title 3. Autocorrelação, comparação de testes, FDR, concordância/kappa
rb = RES["base"]
# --- crítica 1: autocorrelação lag-1 dos resíduos de Sen --------------------------
ac = rb.groupby("city")["r1_resid"].agg(
    r1_median="median", pct_r1_gt=lambda s: 100 * (s > R1_THRESHOLD).mean(), n="size")
ac.loc["ALL"] = [rb.r1_resid.median(), 100 * (rb.r1_resid > R1_THRESHOLD).mean(), len(rb)]
ac.to_csv(f"{OUT_DIR}/autocorrelation_by_unit.csv")
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(rb.r1_resid.dropna(), bins=40, color="0.5")
ax.axvline(R1_THRESHOLD, color="crimson", ls="--")
ax.set_xlabel("lag-1 autocorrelation of Sen residuals"); ax.set_ylabel("series")
fig.tight_layout(); fig.savefig(f"{OUT_DIR}/fig_r1_hist.png", dpi=200); plt.close(fig)

# --- % significativo por teste e perda em p.p. ----------------------------------------
def sig_table(res):
    t = pd.DataFrame({f"pct_sig_{k}": res.groupby("city")[f"class_{k}"].apply(
        lambda s: pct(np.where(s.isna(), np.nan, s != 0))) for k in TESTS})
    t.loc["ALL"] = [pct(np.where(res[f"class_{k}"].isna(), np.nan, res[f"class_{k}"] != 0))
                    for k in TESTS]
    for k in ["mk_ties", "hr", "tfpw", "hr_fdr"]:
        t[f"loss_pp_mk_to_{k}"] = t["pct_sig_mk"] - t[f"pct_sig_{k}"]
    return t
SIG = {v: sig_table(r) for v, r in RES.items()}
SIG["base"].to_csv(f"{OUT_DIR}/tests_by_unit.csv")

# --- concordância e kappa entre testes (base) e entre versões (teste principal) --------
rows = []
for i, a in enumerate(TESTS):
    for b in TESTS[i + 1:]:
        po, kap, n = cohen_kappa(rb[f"class_{a}"], rb[f"class_{b}"])
        rows.append({"comparison": f"{a} vs {b}", "scope": "ALL", "agreement_pct": 100 * po,
                     "kappa": kap, "n": n})
        for c, g in rb.groupby("city"):
            po, kap, n = cohen_kappa(g[f"class_{a}"], g[f"class_{b}"])
            rows.append({"comparison": f"{a} vs {b}", "scope": c, "agreement_pct": 100 * po,
                         "kappa": kap, "n": n})
for v in VARIANT_KEYS:
    j = rb[["city", f"class_{PRINCIPAL}"]].join(RES[v][[f"class_{PRINCIPAL}"]], rsuffix="_v",
                                                 how="inner")
    po, kap, n = cohen_kappa(j[f"class_{PRINCIPAL}"], j[f"class_{PRINCIPAL}_v"])
    rows.append({"comparison": f"base vs {v} ({PRINCIPAL})", "scope": "ALL",
                 "agreement_pct": 100 * po, "kappa": kap, "n": n})
    for c, g in j.groupby("city"):
        po, kap, n = cohen_kappa(g[f"class_{PRINCIPAL}"], g[f"class_{PRINCIPAL}_v"])
        rows.append({"comparison": f"base vs {v} ({PRINCIPAL})", "scope": c,
                     "agreement_pct": 100 * po, "kappa": kap, "n": n})
AGREE = pd.DataFrame(rows)
AGREE.to_csv(f"{OUT_DIR}/agreement_kappa.csv", index=False)
print(ac.round(3).tail(1).to_string())
print(SIG["base"].loc["ALL"].round(2).to_string())
print(AGREE[AGREE.scope == "ALL"].round(3).to_string(index=False))

In [ ]:
# @title 4. Crítica 3: observações por ano/unidade, trocas de sensor e Pettitt
d0 = PANELS["base"]
nobs = (pd.concat({v: d.groupby(["city", "year"])[["n_obs", "n_TM", "n_ETM", "n_OLI"]].median()
                   for v, d in PANELS.items()}, names=["version"]).reset_index())
nobs.to_csv(f"{OUT_DIR}/nobs_by_unit_year.csv", index=False)

# anos de troca de sensor detectados nos DADOS: muda a combinação modal de sensores
mode_flag = d0.groupby("year")["sensor_flag"].agg(lambda s: s.mode().iloc[0] if len(s.mode()) else np.nan)
SENSOR_NAMES = {1: "TM", 2: "ETM+", 4: "OLI", 8: "OLI-2"}
flag_name = lambda f: "+".join(n for b, n in SENSOR_NAMES.items() if int(f) & b) if pd.notna(f) else "?"
mode_names = mode_flag.map(flag_name)
CHANGE_YEARS = [int(y) for y, prev in zip(mode_names.index[1:], mode_names.values[:-1])
                if mode_names[y] != prev]
print("Combinação modal de sensores por ano:", mode_names.to_dict())
print("Anos em que a combinação muda:", CHANGE_YEARS)

rows = []
for v, P in SERIES.items():
    rv = RES[v]
    meta = PANELS[v].drop_duplicates("pid").set_index("pid")
    for c in CITIES:
        ids = rv.index[rv.city == c]
        subsets = {"all": ids}
        if "mb_stable" in rv:
            subsets["stable"] = rv.index[(rv.city == c) & (rv.mb_stable == 1)]
        for sname, sid in subsets.items():
            if len(sid) == 0:
                continue
            med = P.loc[sid].median(axis=0).to_numpy(float)
            yr, p = pettitt(YEARS, med)
            rows.append({"version": v, "city": c, "subset": sname, "n_pixels": len(sid),
                         "break_year": yr, "p": p, "sig": p < ALPHA_SIG,
                         "break_at_sensor_change": bool(p < ALPHA_SIG and yr in CHANGE_YEARS)})
PETT = pd.DataFrame(rows)
PETT.to_csv(f"{OUT_DIR}/pettitt.csv", index=False)
print(PETT.groupby(["version", "subset"])[["sig", "break_at_sensor_change"]].sum().to_string())

In [ ]:
# @title 5. Tabela de sensibilidade consolidada e conclusões robustas
def unit_conclusion(res, key):
    """C1: classe significativa dominante; C2: sinal da mediana do declive de Sen."""
    out = {}
    for c, g in res.groupby("city"):
        cl = g[f"class_{key}"].dropna()
        pos, neg = (cl > 0).mean(), (cl < 0).mean()
        c1 = "increase" if pos > neg else ("decrease" if neg > pos else "none")
        med = g["sen_slope"].median()
        c2 = "positive" if med > 0 else ("negative" if med < 0 else "zero")
        out[c] = f"{c1}|{c2}"
    return pd.Series(out)

VERSIONS_TESTS = [("base", k) for k in TESTS] + [(v, PRINCIPAL) for v in RES if v != "base"]
CONC = pd.DataFrame({f"{v}:{k}": unit_conclusion(RES[v], k) for v, k in VERSIONS_TESTS})
CONC["robust_all_versions"] = CONC.nunique(axis=1).eq(1)
CONC.to_csv(f"{OUT_DIR}/robust_conclusions.csv")

sb = SIG["base"].drop(index="ALL")
tab = pd.DataFrame(index=CITIES)
tab["n_series"] = INFO["base"].n_series
tab["pct_complete"] = 100 * INFO["base"].n_complete / INFO["base"].n_series
tab["r1_median"] = ac["r1_median"]
tab[f"pct_r1_gt_{R1_THRESHOLD}"] = ac["pct_r1_gt"]
for k in TESTS:
    tab[f"pct_sig_{k}"] = sb[f"pct_sig_{k}"]
for v in [x for x in RES if x != "base"]:
    tab[f"pct_sig_{PRINCIPAL}_{v}"] = SIG[v].drop(index="ALL")[f"pct_sig_{PRINCIPAL}"]
for k in ["hr", "tfpw", "hr_fdr"]:
    tab[f"loss_pp_mk_to_{k}"] = sb[f"loss_pp_mk_to_{k}"]
A = AGREE.set_index(["comparison", "scope"])
for comp, col_ in [("mk vs hr", "kappa_mk_vs_hr"), ("hr vs tfpw", "kappa_hr_vs_tfpw")] + \
        [(f"base vs {v} ({PRINCIPAL})", f"kappa_base_vs_{v}") for v in VARIANT_KEYS]:
    tab[col_] = [A.loc[(comp, c), "kappa"] if (comp, c) in A.index else np.nan for c in CITIES]
tab["median_sen_slope"] = RES["base"].groupby("city")["sen_slope"].median()
tab["median_n_obs"] = nobs[nobs.version == "base"].groupby("city")["n_obs"].median()
if TREND_SOURCE == "filled":
    tab["pct_series_with_filled_year"] = 100 * (1 - INFO["base"].n_nofill / INFO["base"].n_complete)
    tab["median_n_filled_years"] = RES["base"].groupby("city")["n_filled"].median()
    tab["max_gap_years"] = RES["base"].groupby("city")["max_gap"].max()
for v in RES:
    pv = PETT[(PETT.version == v) & (PETT.subset == "all")].set_index("city")
    tab[f"pettitt_year_{v}"] = pv["break_year"]
    tab[f"pettitt_p_{v}"] = pv["p"]
tab["conclusion_base_principal"] = CONC[f"base:{PRINCIPAL}"]
tab["robust_all_versions"] = CONC["robust_all_versions"]
tab.to_csv(f"{OUT_DIR}/sensitivity_table.csv")
print(tab.round(3).to_string())

In [ ]:
# @title 6. Arquivos e figuras das classes de tendência (cada teste e variante)
CLASS_COLORS = {-1: "#b2182b", 0: "#d9d9d9", 1: "#2166ac"}
for v, res in RES.items():
    feats = []
    for pid, r in res.iterrows():
        props = {"pid": pid, "city": r.city, "sen_slope": r.sen_slope, "tau": r.tau,
                 "r1_resid": r.r1_resid}
        props.update({f"class_{k}": (None if pd.isna(r[f"class_{k}"]) else int(r[f"class_{k}"]))
                      for k in TESTS})
        props.update({f"p_{k}": (None if pd.isna(r[f"p_{k}"]) else float(r[f"p_{k}"]))
                      for k in ["mk", "mk_ties", "hr", "tfpw"]})
        feats.append({"type": "Feature", "properties": props,
                      "geometry": {"type": "Point", "coordinates": [r.lon, r.lat]}})
    with open(f"{OUT_DIR}/classes_points_{v}.geojson", "w") as f:
        json.dump({"type": "FeatureCollection", "features": feats}, f)
    keys = TESTS if v == "base" else [PRINCIPAL]
    for k in keys:
        nc = len(CITIES)
        ncol = 7
        fig, axes = plt.subplots(math.ceil(nc / ncol), ncol, figsize=(2.4 * ncol, 2.4 * math.ceil(nc / ncol)))
        for ax, c in zip(np.ravel(axes), CITIES):
            g = res[res.city == c]
            ax.scatter(g.lon, g.lat, s=4, c=[CLASS_COLORS.get(x, "white") for x in g[f"class_{k}"]])
            ax.set_title(c, fontsize=7); ax.set_xticks([]); ax.set_yticks([])
        for ax in np.ravel(axes)[nc:]:
            ax.axis("off")
        fig.suptitle(f"{TEST_LABEL[k]} — {v} (blue = significant increase, red = decrease)")
        fig.tight_layout(); fig.savefig(f"{OUT_DIR}/fig_classes_{v}_{k}.png", dpi=200); plt.close(fig)
print("GeoJSON e figuras em", OUT_DIR)

In [ ]:
# @title 7. Texto em inglês (números gerados a partir das saídas acima)
S = {}
S["n_units"], S["n_years"] = len(CITIES), len(YEARS)
S["n_series"] = int(INFO["base"].n_series.sum()); S["n_complete"] = len(RES["base"])
S["pct_complete"] = 100 * S["n_complete"] / S["n_series"]
S["filled"] = TREND_SOURCE == "filled" and "fill_type" in PANELS["base"]
if S["filled"]:
    _ft = PANELS["base"]["fill_type"]
    S["pct_py_filled"], S["pct_py_interp"] = 100 * (_ft > 0).mean(), 100 * (_ft == 1).mean()
    S["pct_py_edge"], S["pct_py_spatial"] = 100 * (_ft == 2).mean(), 100 * (_ft == 3).mean()
    S["pct_py_city"] = 100 * (_ft == 4).mean()
    S["n_nofill"] = len(RES.get("base_nofill", []))
    S["pct_nofill"] = 100 * S["n_nofill"] / max(S["n_complete"], 1)
    S["sig_nofill"] = (float(SIG["base_nofill"].loc["ALL", f"pct_sig_{PRINCIPAL}"])
                       if "base_nofill" in SIG else float("nan"))
FILL_TXT = (f" Missing pixel-years (cloud masking, fewer than {MIN_OBS} clear observations, or missing heat "
            "or aerosol values) were then filled in time, pixel by pixel: by linear interpolation between the "
            "nearest preceding and following years with data (the mean of the two for a one-year gap); at the "
            "start or end of the record, by the nearest year with data; for pixels never observed, by the mean of "
            f"the pixels with data in the same year within a {WIN_RADIUS_M / 1000:g}-km window; and, where that window "
            "had no data, by the unit median of that year. All continuous bands (index, components and raw "
            "descriptors) were filled in the same way. Every value carries a fill flag and the length of the bridged "
            f"gap. In the sampled panel, {S['pct_py_filled']:.1f}% of pixel-years were filled ({S['pct_py_interp']:.1f}% "
            f"interpolated, {S['pct_py_edge']:.1f}% at the ends of the record, {S['pct_py_spatial']:.2f}% from the "
            f"spatial window and {S['pct_py_city']:.2f}% from the unit median)."
            + (" ERA5-Land cells without data (over the sea) took the mean of the adjacent land cells."
               if HEAT_FILL_NEIGHBORS else "") if S["filled"] else "")
SERIES_TXT = (f"using the gap-filled {S['n_years']}-year series ({S['n_complete']} of {S['n_series']} sampled "
              f"pixels; {S['n_nofill']} of them, {S['pct_nofill']:.1f}%, without any filled year)"
              if S["filled"] else
              f"restricted to complete {S['n_years']}-year series ({S['n_complete']} of {S['n_series']} sampled "
              f"pixels, {S['pct_complete']:.1f}%)")
MISSING_TXT = (f"Because the series were gap-filled, all four tests were applied to every pixel; as a sensitivity "
               f"check, all results were recomputed on the pixels without any filled year, where "
               f"{S['sig_nofill']:.1f}% of the series showed a significant trend under the principal test, against "
               f"{float(SIG['base'].loc['ALL', 'pct_sig_' + PRINCIPAL]):.1f}% in the full sample."
               if S["filled"] else
               "Missing pixel-years were excluded from tests (a) and (b) in the maps; tests (c) and (d), which "
               "require equally spaced data, were applied only to complete series.")
S["r1_med"] = float(ac.loc["ALL", "r1_median"]); S["r1_gt"] = float(ac.loc["ALL", "pct_r1_gt"])
S["r1_unit_min"] = float(ac.drop(index="ALL").r1_median.min())
S["r1_unit_max"] = float(ac.drop(index="ALL").r1_median.max())
sa = SIG["base"].loc["ALL"]
for k in TESTS:
    S[f"sig_{k}"] = float(sa[f"pct_sig_{k}"])
S["loss_hr"] = float(sa["loss_pp_mk_to_hr"]); S["loss_tfpw"] = float(sa["loss_pp_mk_to_tfpw"])
S["loss_fdr"] = float(sa["pct_sig_hr"] - sa["pct_sig_hr_fdr"])
lu = SIG["base"].drop(index="ALL")["loss_pp_mk_to_hr"]
S["loss_hr_unit_min"], S["loss_hr_unit_max"] = float(lu.min()), float(lu.max())
AA = AGREE[AGREE.scope == "ALL"].set_index("comparison")
S["agr_mk_hr"], S["kap_mk_hr"] = AA.loc["mk vs hr", "agreement_pct"], AA.loc["mk vs hr", "kappa"]
S["agr_hr_tfpw"], S["kap_hr_tfpw"] = AA.loc["hr vs tfpw", "agreement_pct"], AA.loc["hr vs tfpw", "kappa"]
S["n_robust"] = int(CONC["robust_all_versions"].sum())
S["robust_units"] = ", ".join(CONC.index[CONC.robust_all_versions]) or "none"
S["nonrobust_units"] = ", ".join(CONC.index[~CONC.robust_all_versions]) or "none"
nb = nobs[nobs.version == "base"]
S["nobs_min"], S["nobs_max"] = float(nb.n_obs.min()), float(nb.n_obs.max())
yr_med = nb.groupby("year").n_obs.median()
S["nobs_year_min"], S["nobs_year_min_val"] = int(yr_med.idxmin()), float(yr_med.min())
S["nobs_year_max"], S["nobs_year_max_val"] = int(yr_med.idxmax()), float(yr_med.max())
pb = PETT[(PETT.version == "base") & (PETT.subset == "all")]
S["pett_sig"], S["pett_sensor"] = int(pb.sig.sum()), int(pb.break_at_sensor_change.sum())
has_noharm, has_season = "noharm" in RES, "season" in RES
if has_noharm:
    pn = PETT[(PETT.version == "noharm") & (PETT.subset == "all")]
    S["pett_sensor_noharm"] = int(pn.break_at_sensor_change.sum())
    S["kap_noharm"] = AA.loc[f"base vs noharm ({PRINCIPAL})", "kappa"]
    S["agr_noharm"] = AA.loc[f"base vs noharm ({PRINCIPAL})", "agreement_pct"]
    S["sig_noharm"] = float(SIG["noharm"].loc["ALL", f"pct_sig_{PRINCIPAL}"])
if has_season:
    S["kap_season"] = AA.loc[f"base vs season ({PRINCIPAL})", "kappa"]
    S["agr_season"] = AA.loc[f"base vs season ({PRINCIPAL})", "agreement_pct"]
    S["sig_season"] = float(SIG["season"].loc["ALL", f"pct_sig_{PRINCIPAL}"])
    S["season_counts"] = season_tab.season.value_counts().to_dict() if "season_tab" in globals() else {}
ps = PETT[(PETT.version == "base") & (PETT.subset == "stable")] if "stable" in set(PETT.subset) else None
S["pett_stable_sensor"] = int(ps.break_at_sensor_change.sum()) if ps is not None else None
S["max_diff_pmk"] = float(VALIDATION[VALIDATION.test.isin(["mk_ties", "hr", "tfpw"])].max_abs_diff_p.max())
json.dump({k: (v if not isinstance(v, (np.floating, np.integer)) else v.item()) for k, v in S.items()},
          open(f"{OUT_DIR}/text_numbers.json", "w"), indent=1, default=str)

f1 = lambda x: f"{x:.1f}"; f2 = lambda x: f"{x:.2f}"

def chg(loss):
    """Frase neutra para a variação do % significativo (loss > 0 = perda)."""
    if loss > 0:
        return f"reduced the share of significant pixels by {loss:.1f} percentage points"
    if loss < 0:
        return f"increased the share of significant pixels by {-loss:.1f} percentage points"
    return "left the share of significant pixels unchanged"

def _ord(k):
    k = int(k)
    suf = "th" if 10 <= k % 100 <= 20 else {1: "st", 2: "nd", 3: "rd"}.get(k % 10, "th")
    return f"{k}{suf}"

du = -lu                                     # variação por unidade (HR − MK), em p.p.
S["d_hr_unit_min"], S["d_hr_unit_max"] = float(du.min()), float(du.max())
h0, h1 = HEAT_LOCAL_HOURS
window = "the full calendar year" if COMPOSITE_MONTHS is None else f"months {COMPOSITE_MONTHS[0]}–{COMPOSITE_MONTHS[1]}"
l7 = ("Landsat 7 ETM+ was used only in years without OLI data (up to 2012)" if L7_POLICY == "no_oli_years"
      else "Landsat 7 ETM+ was used in all years")
harm = (f"TM and ETM+ reflectance was converted to OLI-equivalent values with the {ROY_METHOD} "
        "transformation of Roy et al. (2016)" if HARMONIZE else "no cross-sensor harmonization was applied")

INTRO = (
"Because the index series are short (n = {n} annual values), may be non-normal and can contain outliers, "
"we assess monotonic change with rank-based, distribution-free methods: the Mann–Kendall test and Sen's slope "
"estimator. These methods are standard in environmental trend analysis but are sensitive to serial correlation "
"and to inhomogeneities in the series; Section 2.X describes how both issues were tested and corrected."
).format(n=S["n_years"])

METHODS = f"""### 2.X Annual compositing
Each annual value of HU-EEQI was derived from the per-pixel median of all clear Landsat Collection 2 Level-2 surface-reflectance observations acquired within {window} (Landsat 5 TM, Landsat 7 ETM+, Landsat 8 OLI and Landsat 9 OLI-2). {l7}. Observations were kept only when QA_PIXEL bits 0–4 (fill, dilated cloud, cirrus, cloud and cloud shadow) were zero, no band was saturated (QA_RADSAT = 0) and all six reflective bands lay between 0 and 1; surface temperature was additionally screened with ST_QA ≤ {ST_QA_MAX_K} K. Pixel-years with fewer than {MIN_OBS} clear observations were set to missing.{FILL_TXT} {harm[0].upper() + harm[1:]}. The heat component used ERA5-Land hourly 2-m air and dew-point temperature ({'NWS heat index' if HEAT_METHOD == 'nws_hi' else 'Steadman apparent temperature'}), taking the daily maximum between {h0}:00 and {h1}:00 local time and its annual {'median' if HEAT_STAT == 'median_daily_max' else '95th percentile'}. Aerosol optical depth was the annual median of quality-screened MAIAC retrievals. All layers were used at their native resolution (Landsat 30 m, ERA5-Land about 9 km, MAIAC 1 km, GHS-POP 100 m), without reprojection, aggregation or interpolation, and the index was computed pixel by pixel on the 30-m export grid. The number of clear observations per pixel-year is stored with each composite (median per unit and year in Table S_nobs).

### 2.Y Trend detection
Trends were tested with the Mann–Kendall (MK) statistic S = Σ_(i<j) sgn(x_j − x_i), whose variance under the null hypothesis of no trend and independent observations is Var(S) = [n(n−1)(2n+5) − Σ_g t_g(t_g−1)(2t_g+5)]/18, where t_g is the size of the g-th group of tied values. The standardized statistic is Z = (S−1)/√Var(S) for S > 0, 0 for S = 0 and (S+1)/√Var(S) for S < 0, and two-sided p-values were obtained from the standard normal distribution. The test assumes serially independent observations and a monotonic trend; it does not assume normality. Trend magnitude was estimated with Sen's slope, the median of (x_j − x_i)/(t_j − t_i) over all pairs i < j, with intercept equal to the median of x_i − slope·t_i.

To account for serial correlation, four variants were computed for every series: (a) MK without tie correction; (b) MK with tie correction; (c) the variance-corrected MK of Hamed and Rao (1998), in which Var(S) from (b) is multiplied by n/n*_s = 1 + 2/[n(n−1)(n−2)] Σ_k (n−k)(n−k−1)(n−k−2) ρ_k, ρ_k being the lag-k autocorrelation of the ranks of the series after removing Sen's slope, summed only over lags with |ρ_k| > z_(1−α/2)/√n; and (d) trend-free pre-whitening (TFPW; Yue and Wang, 2002), in which the Sen trend is removed, the lag-1 autoregressive component of the residuals is filtered out, the trend is added back and MK is applied to the n−1 resulting values. Test (c) was adopted as the principal test because it corrects the variance of S for autocorrelation at all significant lags without modifying the series itself, whereas pre-whitening alters the data and was therefore used as a sensitivity test. Serial dependence was quantified as the lag-1 autocorrelation of the residuals from Sen's slope.

Each pixel was classified as significant increase (p < {ALPHA_SIG} and S > 0), significant decrease (p < {ALPHA_SIG} and S < 0) or no significant trend. Because every pixel is tested separately, the false discovery rate was controlled with the Benjamini–Hochberg procedure (Benjamini and Hochberg, 1995) at q = {FDR_Q} within each unit, applied to the p-values of the principal test. Agreement between classifications was measured as the percentage of identical classes and Cohen's kappa (Cohen, 1960).

### 2.Z Homogeneity and robustness
Change points in the unit-median series were tested with the Pettitt (1979) test, and breaks were flagged when they coincided with a change in the dominant Landsat sensor combination detected in the data ({', '.join(map(str, CHANGE_YEARS)) or 'none'}). {'The same test was applied to pixels whose MapBiomas land-cover class did not change between ' + str(MB_YEARS[0]) + ' and ' + str(MB_YEARS[1]) + ', where no real change is expected. ' if S['pett_stable_sensor'] is not None else ''}Two alternative series were recomputed at the sampled pixels with the same calibration: one without cross-sensor harmonization and one composited only within a fixed season, defined for each unit as the standard three-month season (DJF, MAM, JJA or SON) with the lowest mean ERA5-Land precipitation over {YEAR_START}–{YEAR_END} (Muñoz-Sabater et al., 2021).

### 2.W Implementation
Per-pixel maps of all four tests were computed in Google Earth Engine (Gorelick et al., 2017) with the Python API (earthengine-api {VERS['earthengine-api']}). The comparison between tests and series variants used a stratified random sample of {N_PANEL_PER_CITY} pixels per unit (seed {PANEL_SEED} + unit index) {SERIES_TXT}, and was carried out a posteriori in Python {VERS['python']} (NumPy {VERS['numpy']}, pandas {VERS['pandas']}). Our vectorized implementation reproduced pyMannKendall {VERS['pymannkendall']} (Hussain and Mahmud, 2019) to within {S['max_diff_pmk']:.1e} in p. {MISSING_TXT} Significance level: α = {ALPHA_SIG}.

The contribution of this analysis does not rest on the statistical methods, which are standard, but on applying them consistently to {S['n_units']} state capitals over a {S['n_years']}-year Landsat record, on a standardized comparison between units, and on testing the robustness of the conclusions to serial correlation, multiple testing, seasonal compositing and sensor changes.
"""

def _opt(flag, text):
    return text if flag else ""

RESULTS = f"""### 3.X Robustness of the trend results
The median lag-1 autocorrelation of the Sen residuals was {f2(S['r1_med'])} (unit medians from {f2(S['r1_unit_min'])} to {f2(S['r1_unit_max'])}), and {f1(S['r1_gt'])}% of the series exceeded {R1_THRESHOLD} (Table S_ac; Fig. S_r1). The share of pixels with a significant trend was {f1(S['sig_mk'])}% with the original MK test, {f1(S['sig_mk_ties'])}% with tie correction, {f1(S['sig_hr'])}% with the Hamed–Rao correction (principal test), {f1(S['sig_tfpw'])}% with TFPW and {f1(S['sig_hr_fdr'])}% after Benjamini–Hochberg control of the false discovery rate. Relative to the original test, the Hamed–Rao correction {chg(S['loss_hr'])} overall (unit-level change from {S['d_hr_unit_min']:+.1f} to {S['d_hr_unit_max']:+.1f} points) and TFPW {chg(S['loss_tfpw'])}; FDR control, applied to the Hamed–Rao p-values, {chg(S['loss_fdr'])} (Table S_sens). Pixel classes agreed in {f1(S['agr_mk_hr'])}% of cases between the original and Hamed–Rao tests (κ = {f2(S['kap_mk_hr'])}) and in {f1(S['agr_hr_tfpw'])}% between Hamed–Rao and TFPW (κ = {f2(S['kap_hr_tfpw'])}).{_opt(has_season, f" Compositing within a fixed dry season changed the share of significant pixels to {f1(S.get('sig_season', float('nan')))}% (agreement with the annual series {f1(S.get('agr_season', float('nan')))}%, κ = {f2(S.get('kap_season', float('nan')))}).")}{_opt(has_noharm, f" Without cross-sensor harmonization the share was {f1(S.get('sig_noharm', float('nan')))}% (agreement {f1(S.get('agr_noharm', float('nan')))}%, κ = {f2(S.get('kap_noharm', float('nan')))}).")} The Pettitt test detected a significant change point in {S['pett_sig']} of {S['n_units']} unit-median series, {S['pett_sensor']} of them in a year of sensor change{_opt(has_noharm, f" ({S.get('pett_sensor_noharm')} without harmonization)")}. The unit-level conclusions (dominant significant class and sign of the median Sen slope) were identical across all tests and series variants in {S['n_robust']} of {S['n_units']} units ({S['robust_units']}); they differed in {S['nonrobust_units']} (Table S_conc). All trends are reported as associations over time and are not interpreted as evidence of causal effects.
"""

_sensor_tail = ""
if S["pett_stable_sensor"] is not None:
    _sensor_tail += f", {S['pett_stable_sensor']} in land-cover-stable areas"
if has_noharm:
    _sensor_tail += (f"; without harmonization, {S['pett_sensor_noharm']} unit series showed such a "
                     f"break and pixel classes agreed with the harmonized series in "
                     f"{f1(S['agr_noharm'])}% of cases (κ = {f2(S['kap_noharm'])})")
LIMITS = f"""Several limitations should be considered. First, serial correlation: the median lag-1 autocorrelation of the residuals from Sen's slope was {f2(S['r1_med'])} and {f1(S['r1_gt'])}% of the series exceeded {R1_THRESHOLD}; relative to the original MK test, the Hamed–Rao correction {chg(S['loss_hr'])}, and only the corrected results are interpreted. Second, HU-EEQI is partly relative: descriptors are normalized with bounds set at the {_ord(BOUNDS_PCT[0])} and {_ord(BOUNDS_PCT[1])} percentiles pooled across all units and years, the equity component compares each pixel with a {WIN_RADIUS_M / 1000:g}-km neighbourhood and the vulnerability term is rescaled within each unit, so changes describe position within this reference frame rather than absolute environmental quality. Third, {'composites span the full calendar year, so the dates contributing to each annual value are not controlled' if COMPOSITE_MONTHS is None else 'composites are restricted to a fixed window'}{_opt(has_season, f'; restricting compositing to a fixed dry season yielded {f1(S.get("sig_season", float("nan")))}% significant pixels against {f1(S["sig_hr"])}% for the annual series (κ = {f2(S.get("kap_season", float("nan")))})')}. Fourth, the number of clear observations varied between {f1(S['nobs_min'])} and {f1(S['nobs_max'])} per unit-year (median across units lowest in {S['nobs_year_min']}, {f1(S['nobs_year_min_val'])}, and highest in {S['nobs_year_max']}, {f1(S['nobs_year_max_val'])}), so the precision of annual values is not uniform. Fifth, the record combines Landsat sensors (the dominant combination changed in {', '.join(map(str, CHANGE_YEARS)) or 'no year'}); {'with' if HARMONIZE else 'without'} cross-sensor harmonization, {S['pett_sensor']} of {S['n_units']} unit-median series showed a significant change point in a year of sensor change{_sensor_tail}. Finally, the trends describe temporal associations and do not identify their causes.
"""

REFS = """Benjamini, Y., Hochberg, Y. (1995). Controlling the false discovery rate: a practical and powerful approach to multiple testing. J. R. Stat. Soc. B 57, 289–300.
Cohen, J. (1960). A coefficient of agreement for nominal scales. Educ. Psychol. Meas. 20, 37–46.
Gorelick, N. et al. (2017). Google Earth Engine: planetary-scale geospatial analysis for everyone. Remote Sens. Environ. 202, 18–27.
Hamed, K.H., Rao, A.R. (1998). A modified Mann-Kendall trend test for autocorrelated data. J. Hydrol. 204, 182–196.
Hussain, M.M., Mahmud, I. (2019). pyMannKendall: a python package for non parametric Mann Kendall family of trend tests. J. Open Source Softw. 4(39), 1556.
Kendall, M.G. (1975). Rank Correlation Methods. Griffin, London.
Mann, H.B. (1945). Nonparametric tests against trend. Econometrica 13, 245–259.
Muñoz-Sabater, J. et al. (2021). ERA5-Land: a state-of-the-art global reanalysis dataset for land applications. Earth Syst. Sci. Data 13, 4349–4383.
Pettitt, A.N. (1979). A non-parametric approach to the change-point problem. Appl. Stat. 28, 126–135.
Roy, D.P. et al. (2016). Characterization of Landsat-7 to Landsat-8 reflective wavelength and normalized difference vegetation index continuity. Remote Sens. Environ. 185, 57–70.
Sen, P.K. (1968). Estimates of the regression coefficient based on Kendall's tau. J. Am. Stat. Assoc. 63, 1379–1389.
Yue, S., Wang, C.Y. (2002). Applicability of prewhitening to eliminate the influence of serial correlation on the Mann-Kendall test. Water Resour. Res. 38(6), 1068.
"""

with open(f"{OUT_DIR}/manuscript_text_EN.md", "w", encoding="utf-8") as f:
    f.write("## Introduction (single paragraph)\n\n" + INTRO + "\n\n## Methods\n\n" + METHODS +
            "\n## Results\n\n" + RESULTS + "\n## Limitations\n\n" + LIMITS +
            "\n## References (check against the final reference list)\n\n" + REFS)
print(INTRO, METHODS, RESULTS, LIMITS, sep="\n\n")

In [ ]:
# @title 8. Respostas ao revisor (uma por crítica), com a seção/tabela/figura de suporte
RESP = f"""**Comment 1 — Serial autocorrelation.** We thank the reviewer. Serial dependence was quantified for every sampled series as the lag-1 autocorrelation of the residuals from Sen's slope (median {f2(S['r1_med'])}; {f1(S['r1_gt'])}% of series above {R1_THRESHOLD}; Table S_ac, Fig. S_r1). We now report four versions of the test — original MK, tie-corrected MK, the variance-corrected test of Hamed and Rao (1998) and TFPW (Yue and Wang, 2002) — and adopt Hamed–Rao as the principal test (Section 2.Y). Relative to the original test, the Hamed–Rao correction {chg(S['loss_hr'])} and TFPW {chg(S['loss_tfpw'])}; class agreement with the original test was {f1(S['agr_mk_hr'])}% (κ = {f2(S['kap_mk_hr'])}) (Section 3.X; Tables S_tests, S_kappa). All results in the main text are now based on the corrected test. The comparison was performed a posteriori on a stratified random sample of {N_PANEL_PER_CITY} pixels per unit (seed {PANEL_SEED} + unit index), and the corrected tests were also mapped for all pixels (Fig. S_maps).

**Comment 2 — Multiple testing.** Because each pixel is tested independently, we now control the false discovery rate with the Benjamini–Hochberg procedure at q = {FDR_Q} within each unit (Section 2.Y). FDR control reduced the share of significant pixels from {f1(S['sig_hr'])}% to {f1(S['sig_hr_fdr'])}% (Table S_sens; FDR thresholds per unit in Table S_fdr).

**Comment 3 — Seasonality and homogeneity.** We now describe exactly how each annual value was composited — statistic, date window and quality filters (Section 2.X) — and report the number of clear observations per year and unit (Table S_nobs; range {f1(S['nobs_min'])}–{f1(S['nobs_max'])}). {('A fixed-season composite (driest standard three-month season per unit, from ERA5-Land precipitation) gave ' + f1(S.get('sig_season', float('nan'))) + '% significant pixels and κ = ' + f2(S.get('kap_season', float('nan'))) + ' relative to the annual series. ') if has_season else 'A fixed-season composite could not be evaluated and is acknowledged as a limitation. '}Cross-sensor reflectance was harmonized ({ROY_METHOD} coefficients of Roy et al., 2016); the Pettitt test found {S['pett_sensor']} of {S['n_units']} unit series with a significant break in a year of sensor change{(' (' + str(S.get('pett_sensor_noharm')) + ' without harmonization)') if has_noharm else ''}{(', and ' + str(S['pett_stable_sensor']) + ' in land-cover-stable areas') if S['pett_stable_sensor'] is not None else ''} (Section 2.Z; Table S_pettitt).

**Comment 4 — Presentation of the method.** The formal description of MK and Sen's slope (S, Var(S), Z and assumptions) has been moved from the Introduction to the Methods (Section 2.Y); the Introduction now keeps a single paragraph motivating non-parametric tests. Section 2.W details the implementation (Google Earth Engine and Python with package versions, treatment of ties and missing values, α = {ALPHA_SIG} and the classification rule) and frames the contribution in terms of scale, record length, standardized comparison between units and robustness testing rather than the methods themselves.

**Comment 5 — Accuracy and limitations.** All numbers in the text, tables and figures are now generated by script from the final outputs (repository files *_v6.ipynb; numbers in text_numbers.json). Table S_sens lists, for each unit, the results of the original MK, tie-corrected, Hamed–Rao, TFPW, FDR and the series variants, and Table S_conc marks which unit-level conclusions hold in all versions ({S['n_robust']} of {S['n_units']} units). The Limitations paragraph now quantifies autocorrelation, the relative nature of the index, seasonality, the variation in the number of observations and sensor changes, and all trends are interpreted as associations rather than causal relationships.
"""
with open(f"{OUT_DIR}/reviewer_responses_EN.md", "w", encoding="utf-8") as f:
    f.write(RESP)
print(RESP)
print("Arquivos em", OUT_DIR, ":", sorted(os.listdir(OUT_DIR)))